# One to Rule Them All - ETL

Prerequisites:

Prior to running this make sure toonvert JSON file to ndJSON using colab enterprise: https://pantheon.corp.google.com/vertex-ai/colab/overview?referrer=search&e=13802955&mods=-autopush_coliseum&project=project-bb4fd058-24e7-4ccb-b06&activeNb=projects%2Fproject-bb4fd058-24e7-4ccb-b06%2Flocations%2Fus-central1%2Frepositories%2F4f7cfb9a-cc3a-4d11-a321-90352d1e5ca3


In [32]:
#Imports
from google.cloud import storage
from google.cloud import bigquery
import pandas as pd
import os

In [33]:
# # Test Params
# PROJECT_ID = "project-bb4fd058-24e7-4ccb-b06"  # CDP's Cloud project id
# BUCKET_NAME = "cdp-raw-data-bucket"  # Replace with your bucket name
# BQ_DATASET_RAW = "project-bb4fd058-24e7-4ccb-b06.CSTAR_2025_test_orgs_20260508"  # Replace with your BQ dataset for raw data
# BQ_DATASET_PROCESSED = "project-bb4fd058-24e7-4ccb-b06.CSTAR_2025_processed_test_20260508"  # Replace with your BQ dataset for processed data
# BQ_MODELS_DATASET = BQ_DATASET_PROCESSED

# overture_cdp_geometry_json_file = 'gs://cdp-raw-data-bucket/CSTAR_2025/Overture geometry/geometry_cleaned.ndjson'

In [34]:
# Params
PROJECT_ID = "project-bb4fd058-24e7-4ccb-b06"  # CDP's Cloud project id
BUCKET_NAME = "cdp-raw-data-bucket"  # Replace with your bucket name
BQ_DATASET_RAW = "project-bb4fd058-24e7-4ccb-b06.CSTAR_2025_v2"  # Replace with your BQ dataset for raw data
BQ_DATASET_PROCESSED = "project-bb4fd058-24e7-4ccb-b06.CSTAR_2025_processed_v2"  # Replace with your BQ dataset for processed data
BQ_MODELS_DATASET = BQ_DATASET_PROCESSED

overture_cdp_geometry_json_file = 'gs://cdp-raw-data-bucket/CSTAR_2025/Overture geometry/geometry_cleaned.ndjson'

In [35]:
# Clients
storage_client = storage.Client(project=PROJECT_ID)
bq_client = bigquery.Client(project=PROJECT_ID)

## Central Dim table - initial stage

A following stage will come once the Fact tables are created.

In [36]:
# Construct the SQL query to create the central dimension table

# Define the base table for reporting_org
base_table = f"{BQ_DATASET_RAW}.Summary"

# Define the target central dimension table name
initial_central_dim_table_name = f"{BQ_DATASET_PROCESSED}.dim_central"

# Construct the SQL query

sql_query = f"""
-- create base dim central table
CREATE OR REPLACE TABLE `{initial_central_dim_table_name}` AS
WITH
  -- Create Initial Dimension Table (`dim_central_base`)
  dim_central_base AS (
    SELECT
        s.*,
        q1d1.What_language_are_you_submitting_your_response_in AS reporting_language,
        q1d2.col1_Administrative_boundary_of_reporting_government AS reporting_gov,
        q1d2.col2_Next_highest_level_of_government AS next_high_gov,
        q1d2.col3_Next_lowest_level_of_government AS next_low_gov,
        q1d2.col4_Area_of_the_jurisdiction_boundary_in_square_km AS jdx_areasize,
        q1d2.col5_Percentage_range_of_jurisdiction_area_that_is_natural_or_modified_terrestrial_freshwater_coastal_and_or_marine_ecosystem AS jdx_natural_pct,
        q1d2.col6_Current_or_most_recent_population_size AS current_pop,
        q1d2.col7_Population_year AS cur_pop_year,
        q1d2.col8_Projected_population_size AS proj_pop,
        q1d2.col9_Projected_population_year AS proj_pop_year,
        q1d2.col10_Select_the_currency_used_for_all_financial_information_reported_throughout_your_response AS reporting_currency,
        q1d2.col11_Select_which_Common_Reporting_Framework_level_you_are_reporting_to AS reporting_framework,
        q2d1.Has_a_climate_risk_and_vulnerability_assessment_been_undertaken_for_your_jurisdiction_If_not_please_indicate_why AS climate_assess_yn,
        q5d1.Does_your_jurisdiction_have_an_adaptation_goal_s_in_place_If_no_adaptation_goal_is_in_place_please_indicate_the_primary_reason_why AS adpt_goal_yn,
        q8d1.Does_your_jurisdiction_have_a_climate_action_plan_or_strategy_that_addresses_mitigation_adaptation_resilience_and_or_energy AS action_plan_yn,
        metadata.CHAMP AS champ_yn,
        metadata.GS_GN AS gs_gn,
        metadata.`GDP per capita _US__` AS gdp_pc,
        metadata.`Development status classification` AS dev_status,
        metadata.`Income Group` AS income_group,
        CASE
          WHEN UPPER(q1d2.col10_Select_the_currency_used_for_all_financial_information_reported_throughout_your_response) LIKE '%USD%' THEN 1
          ELSE COALESCE(metadata.FX, 1)
        END AS fx_rate,
        CAST(LEFT(s.disclosure_cycle,4) AS INT64) AS disclosure_year
    FROM
        `{base_table}` AS s
      LEFT JOIN `{BQ_DATASET_RAW}.Q1_1` AS q1d1 ON s.cdp_disclosing_org_number = q1d1.cdp_disclosing_org_number
      LEFT JOIN `{BQ_DATASET_RAW}.Q1_2` AS q1d2 ON s.cdp_disclosing_org_number = q1d2.cdp_disclosing_org_number
      LEFT JOIN `{BQ_DATASET_RAW}.Q2_1` AS q2d1 ON s.cdp_disclosing_org_number = q2d1.cdp_disclosing_org_number
      LEFT JOIN `{BQ_DATASET_RAW}.Q5_1` AS q5d1 ON s.cdp_disclosing_org_number = q5d1.cdp_disclosing_org_number
      LEFT JOIN `{BQ_DATASET_RAW}.Q8_1` AS q8d1 ON s.cdp_disclosing_org_number = q8d1.cdp_disclosing_org_number
      LEFT JOIN `{BQ_DATASET_RAW}.country_metadata` AS metadata ON s.discloser_country_or_area = metadata.`CSTAR Name`
    WHERE
        s.disclosure_status IN ('Submitted', 'Amended')
  )
  select * from dim_central_base;
"""

In [37]:
print(f"Creating central dimension table: {initial_central_dim_table_name}")
print("Executing SQL query:")

# Execute the SQL query
try:
    query_job = bq_client.query(sql_query)
    query_job.result()  # Wait for the query to complete
    print(f"Successfully created central dimension table: {initial_central_dim_table_name}")
except Exception as e:
    print(f"Error creating central dimension table: {e}")

Creating central dimension table: project-bb4fd058-24e7-4ccb-b06.CSTAR_2025_processed_v2.dim_central
Executing SQL query:
Successfully created central dimension table: project-bb4fd058-24e7-4ccb-b06.CSTAR_2025_processed_v2.dim_central


## Fact tables

#### Always-run translation helpers

Run this before any translation table build. It protects acronyms before Cloud Translation and restores the original tokens after translation.

In [38]:
ACRONYM_PROTECTION_SQL = r"""
-- Pure-SQL acronym protection.
--
-- protect_acronyms: wraps acronym-like tokens in <span translate="no">...</span>
--   so Cloud Translate (which honors that HTML attribute) leaves them intact.
--   Fires only for short (LENGTH<=15) single/double-token entries.
--   Longer or multi-word entries are skipped since Translate handles them natively,
--   and the wrap risks breaking words at non-ASCII boundaries (e.g. SOLIDÁRIOS).
--
-- restore_acronyms: strips the <span> wrapping. The `original` parameter is
--   accepted for backward compatibility with prior call sites but unused —
--   the acronym text is preserved literally inside the tags.
CREATE TEMP FUNCTION protect_acronyms(text STRING) AS (
  CASE
    WHEN text IS NULL THEN NULL
    WHEN LENGTH(text) <= 15
         AND ARRAY_LENGTH(SPLIT(text, ' ')) <= 2
         AND REGEXP_CONTAINS(text, r'([A-Z][A-Za-z]*(?:\.[A-Za-z]+){2,}\.?|(?:[A-Za-z]\.){2,}[A-Za-z]?|[A-Z][A-Za-z0-9]*[A-Z0-9][A-Za-z0-9]*(?:[/\-][A-Za-z0-9]+)*)')
    THEN REGEXP_REPLACE(
           text,
           r'([A-Z][A-Za-z]*(?:\.[A-Za-z]+){2,}\.?|(?:[A-Za-z]\.){2,}[A-Za-z]?|[A-Z][A-Za-z0-9]*[A-Z0-9][A-Za-z0-9]*(?:[/\-][A-Za-z0-9]+)*)',
           r'<span translate="no">\1</span>')
    ELSE text
  END
);

CREATE TEMP FUNCTION restore_acronyms(translated STRING, original_text STRING) AS (
  CASE
    WHEN translated IS NULL THEN NULL
    ELSE
      -- Strip span wrapping, then decode common HTML entities that Translate
      -- emits for apostrophes, ampersands, etc. (pre-existing Translate
      -- behavior; cleaning up here improves downstream readability).
      REGEXP_REPLACE(
        REGEXP_REPLACE(
          REGEXP_REPLACE(
            REGEXP_REPLACE(
              REGEXP_REPLACE(translated, r'</?span[^>]*>', ''),
              r'&#39;', "'"),
            r'&amp;', '&'),
          r'&quot;', '"'),
        r'&lt;', '<')
  END
);
"""


HAZARD_FORMAT_SQL = r"""
-- Sentence-case hazard strings while preserving ALL-CAPS acronyms (e.g. "GHG"), collapses
-- capitalization differences for aggregation.
CREATE TEMP FUNCTION restore_caps(formatted STRING, original STRING)
RETURNS STRING
LANGUAGE js AS r'''
  if (formatted === null || original === null) return formatted;
  const caps = original.match(/\b[A-Z]{2,}[A-Za-z0-9]*\b/g) || [];
  let out = formatted;
  for (const cap of caps) {
    const escaped = cap.replace(/[.*+?^${}()|[\]\\]/g, '\\$&');
    out = out.replace(new RegExp('\\b' + escaped + '\\b', 'gi'), cap);
  }
  return out;
''';

CREATE TEMP FUNCTION format_hazard(h STRING) AS (
  restore_caps(
    CASE
      WHEN h IS NULL THEN NULL
      WHEN STARTS_WITH(TRIM(h), 'Other: ') THEN
        CONCAT(
          'Other: ',
          UPPER(SUBSTR(LOWER(SUBSTR(TRIM(h), 8)), 1, 1)),
          SUBSTR(LOWER(SUBSTR(TRIM(h), 8)), 2)
        )
      ELSE
        CONCAT(
          UPPER(SUBSTR(LOWER(TRIM(h)), 1, 1)),
          SUBSTR(LOWER(TRIM(h)), 2)
        )
    END,
    TRIM(h)
  )
);
"""

#### Translation Model

Run once if needed.

In [39]:
translation_model_query = f"""
-- Set up translation model
# https://docs.cloud.google.com/bigquery/docs/translate-text
CREATE OR REPLACE MODEL `{BQ_MODELS_DATASET}.translation_model`
  REMOTE WITH CONNECTION `project-bb4fd058-24e7-4ccb-b06.us.translation-US`
  OPTIONS (REMOTE_SERVICE_TYPE = 'CLOUD_AI_TRANSLATE_V3');
"""

In [40]:
query_job = bq_client.query(translation_model_query)
query_job.result()
print(f"Successfully Set up translation model")

Successfully Set up translation model


#### Remote mode for AI Summary

Run once if needed

In [41]:
AISummery_model_query = f"""
-- Create a remote model for AI summary
CREATE OR REPLACE MODEL `{BQ_MODELS_DATASET}.summary_model`
  REMOTE WITH CONNECTION `projects/project-bb4fd058-24e7-4ccb-b06/locations/us/connections/CDP-GFellowUS`
  OPTIONS (endpoint = 'gemini-2.5-flash');

"""

In [42]:
query_job = bq_client.query(AISummery_model_query)
query_job.result()
print(f"Successfully Set up AI summery model")

Successfully Set up AI summery model


### Fact Goal

In [ ]:
goal_table_name = f"{BQ_DATASET_PROCESSED}.fact_goal"

goal_query = rf"""
{ACRONYM_PROTECTION_SQL}
-- Create Fact Goal Table
CREATE OR REPLACE TABLE `{goal_table_name}` AS
WITH
-- Step 1: Unpivot columns for translation
  Q5_1_1_combined AS (
    SELECT * FROM `{BQ_DATASET_RAW}.Q5_1_1`
    FULL OUTER UNION ALL BY NAME
    SELECT
      * EXCEPT(
        col4_Base_year_of_goal_or_year_goal_was_established_if_no_base_year,
        col5_Target_year_of_goal
      ),
      CAST(col4_Base_year_of_goal_or_year_goal_was_established_if_no_base_year AS FLOAT64)
        AS col4_Base_year_of_goal_or_year_goal_was_established_if_no_base_year,
      CAST(col5_Target_year_of_goal AS FLOAT64) AS col5_Target_year_of_goal
    FROM `Missing_Data.quezon_Q5_1_1`
  ),
  UnpivotedTexts AS (
    SELECT
      disclosure_cycle,
      cdp_disclosing_org_number,
      row_order,
      'goal' AS col_name,
      TRIM(col2_Adaptation_goal) AS text_content
    FROM Q5_1_1_combined
    WHERE col2_Adaptation_goal IS NOT NULL AND TRIM(col2_Adaptation_goal) != ''
    UNION ALL
    SELECT
      disclosure_cycle,
      cdp_disclosing_org_number,
      row_order,
      'hazard_addressed' AS col_name,
      TRIM(col3_Climate_hazards_that_goal_addresses) AS text_content
    FROM Q5_1_1_combined
    WHERE col3_Climate_hazards_that_goal_addresses IS NOT NULL AND TRIM(col3_Climate_hazards_that_goal_addresses)!= ''
    UNION ALL
    SELECT
      disclosure_cycle,
      cdp_disclosing_org_number,
      row_order,
      'metric_used' AS col_name,
      TRIM(col6_Description_of_metric_indicator_used_to_track_goal_and_evidence_of_implementation) AS text_content
    FROM Q5_1_1_combined
    WHERE col6_Description_of_metric_indicator_used_to_track_goal_and_evidence_of_implementation IS NOT NULL AND TRIM(col6_Description_of_metric_indicator_used_to_track_goal_and_evidence_of_implementation)!= ''
    UNION ALL
    SELECT
      disclosure_cycle,
      cdp_disclosing_org_number,
      row_order,
      'comment' AS col_name,
      TRIM(col7_Comment) AS text_content
    FROM Q5_1_1_combined
    WHERE col7_Comment IS NOT NULL AND TRIM(col7_Comment)!= ''
  ),
  -- Step 2: Detect language on unpivoted data
  DetectedLanguages AS (
    SELECT
      ut.disclosure_cycle,
      ut.cdp_disclosing_org_number,
      ut.row_order,
      ut.col_name,
      ut.text_content,
      CASE
        WHEN LENGTH(TRIM(ut.text_content)) <= 9
             AND ARRAY_LENGTH(SPLIT(TRIM(ut.text_content), ' ')) = 1
             AND NOT REGEXP_CONTAINS(ut.text_content, r'[^[:ascii:]]')
             AND LENGTH(REGEXP_REPLACE(ut.text_content, r'[^A-Z]', '')) * 2
                 > LENGTH(REGEXP_REPLACE(ut.text_content, r'[^A-Za-z]', ''))
          THEN 'en'
        ELSE JSON_VALUE(ml_translate_result, '$.languages[0].language_code')
      END AS detected_lang
    FROM ML.TRANSLATE(
            MODEL `{BQ_MODELS_DATASET}.translation_model`,
            TABLE UnpivotedTexts,
            STRUCT('detect_language' AS translate_mode)
         ) AS ut
    WHERE ml_translate_status = ''
  ),
  -- Step 3: Translate non-English texts (acronyms protected via shared SQL UDF)
  Translations AS (
    SELECT
      disclosure_cycle,
      cdp_disclosing_org_number,
      row_order,
      col_name,
      JSON_VALUE(ml_translate_result, '$.translations[0].translated_text') AS translated_text
    FROM ML.TRANSLATE(
      MODEL `{BQ_MODELS_DATASET}.translation_model`,
      (SELECT disclosure_cycle, cdp_disclosing_org_number, row_order, col_name,
              protect_acronyms(text_content) AS text_content
       FROM DetectedLanguages WHERE detected_lang != 'en'),
      STRUCT('translate_text' AS translate_mode, 'en' AS target_language_code)
    )
    WHERE ml_translate_status = ''
  ),
  -- Step 4: Combine original and translated texts (strip acronym wrapping)
  CombinedTranslations AS (
    SELECT
      dl.disclosure_cycle,
      dl.cdp_disclosing_org_number,
      dl.row_order,
      dl.col_name,
      IF(dl.detected_lang = 'en',
         dl.text_content,
         restore_acronyms(tr.translated_text, dl.text_content)
        ) AS english_text
    FROM DetectedLanguages dl
    LEFT JOIN Translations tr ON dl.disclosure_cycle = tr.disclosure_cycle AND dl.cdp_disclosing_org_number = tr.cdp_disclosing_org_number AND dl.row_order = tr.row_order AND dl.col_name = tr.col_name
  ),
  -- Step 5: Pivot translations back into columns
  PivotedTranslations AS (
    SELECT
      disclosure_cycle,
      cdp_disclosing_org_number,
      row_order,
      MAX(IF(col_name = 'goal', english_text, NULL)) AS goal_english,
      MAX(IF(col_name = 'hazard_addressed', english_text, NULL)) AS hazard_addressed_english,
      MAX(IF(col_name = 'metric_used', english_text, NULL)) AS metric_used_english,
      MAX(IF(col_name = 'comment', english_text, NULL)) AS comment_english
    FROM CombinedTranslations
    GROUP BY disclosure_cycle,
      cdp_disclosing_org_number,
      row_order
  ),
  -- Step 6: Append translated fields to the original hazard table
translated_goals as (
SELECT
  t.*,
  pt.goal_english,
  pt.hazard_addressed_english,
  pt.metric_used_english,
  pt.comment_english
FROM Q5_1_1_combined t
LEFT JOIN PivotedTranslations pt
  ON t.disclosure_cycle = pt.disclosure_cycle
  AND t.cdp_disclosing_org_number = pt.cdp_disclosing_org_number
  AND t.row_order = pt.row_order
  ),

  initial_fact_goal as (
    SELECT
    ht.disclosure_cycle,
    ht.cdp_disclosing_org_number,
    ht.disclosing_organization,
    ht.public_status,
    ht.goal_english,
    STRING_AGG(distinct hazard_addressed_english, '|') AS hazard_addressed_english,
    MIN(col4_Base_year_of_goal_or_year_goal_was_established_if_no_base_year) as base_year,
    MAX(col5_Target_year_of_goal) AS target_year,
    STRING_AGG(distinct metric_used_english, '|') AS metric_used_english,
    STRING_AGG(distinct comment_english, '|') AS comment_english,
FROM
    translated_goals ht
GROUP BY 1,2,3,4,5
ORDER BY 1,2,3,4,5
  )

  SELECT
    ht.*,
    CAST(LEFT(disclosure_cycle,4) AS INT64) AS disclosing_year,
    ROW_NUMBER() OVER (PARTITION BY cdp_disclosing_org_number ORDER BY goal_english DESC) AS goal_index
FROM
    initial_fact_goal ht
;
"""


In [44]:
query_job = bq_client.query(goal_query)
query_job.result()
print(f"Successfully created {goal_table_name}")

Successfully created project-bb4fd058-24e7-4ccb-b06.CSTAR_2025_processed_v2.fact_goal


### Fact Action

In [51]:
actions_translation_query = rf"""
{ACRONYM_PROTECTION_SQL}
{HAZARD_FORMAT_SQL}
CREATE OR REPLACE TABLE `{BQ_DATASET_PROCESSED}.action_translated` AS
WITH
  -- Step 1: Unpivot columns for translation
  Q9_1_combined AS (
    SELECT * FROM `{BQ_DATASET_RAW}.Q9_1`
    FULL OUTER UNION ALL BY NAME
    SELECT
      * EXCEPT(
        col14_Total_cost_of_action_in_currency_specified_in_1_2,
        `col9_Proportion_of_the_total_jurisdiction_population_within_the_jurisdiction_boundary_specified_in_1_2_with_increased_resilience_due_to_adaptation_action_%`,
        `col10_Proportion_of_natural_or_modified_terrestrial_freshwater_coastal_and_or_marine_ecosystems_within_the_jurisdiction_boundary_specified_in_1_2_with_increased_resilience_due_to_adaptation_action_%`
      ),
      SAFE_CAST(col14_Total_cost_of_action_in_currency_specified_in_1_2 AS FLOAT64)
        AS col14_Total_cost_of_action_in_currency_specified_in_1_2,
      `col9_Proportion_of_the_total_jurisdiction_population_within_the_jurisdiction_boundary_specified_in_1_2_with_increased_resilience_due_to_adaptation_action_%`
        AS col9_Proportion_of_the_total_jurisdiction_population_within_the_jurisdiction_boundary_specified_in_1_2_with_increased_resilience_due_to_adaptation_action,
      `col10_Proportion_of_natural_or_modified_terrestrial_freshwater_coastal_and_or_marine_ecosystems_within_the_jurisdiction_boundary_specified_in_1_2_with_increased_resilience_due_to_adaptation_action_%`
        AS col10_Proportion_of_natural_or_modified_terrestrial_freshwater_coastal_and_or_marine_ecosystems_within_the_jurisdiction_boundary_specified_in_1_2_with_increased_resilience_due_to_adaptation_action
    FROM `Missing_Data.quezon_Q9_1`
  ),
  UnpivotedTexts AS (
    SELECT
      disclosure_cycle,
      cdp_disclosing_org_number,
      row_order,
      'action' AS col_name,
      TRIM(col2_Action) AS text_content
    FROM Q9_1_combined
    WHERE col2_Action IS NOT NULL AND TRIM(col2_Action) != ''
    --and cdp_disclosing_org_number = 859097 -- Yamato
    UNION ALL
    SELECT
      disclosure_cycle,
      cdp_disclosing_org_number,
      row_order,
      'hazard_addressed' AS col_name,
      TRIM(col3_Climate_hazard_s_that_action_addresses) AS text_content
    FROM Q9_1_combined
    WHERE col3_Climate_hazard_s_that_action_addresses IS NOT NULL AND TRIM(col3_Climate_hazard_s_that_action_addresses)!= ''
    --and cdp_disclosing_org_number = 859097 -- Yamato
    UNION ALL
    SELECT
      disclosure_cycle,
      cdp_disclosing_org_number,
      row_order,
      'action_description' AS col_name,
      TRIM(col4_Action_description_and_web_link_to_further_information) AS text_content
    FROM Q9_1_combined
    WHERE col4_Action_description_and_web_link_to_further_information IS NOT NULL AND TRIM(col4_Action_description_and_web_link_to_further_information)!= ''
    --and cdp_disclosing_org_number = 859097 -- Yamato
    UNION ALL
    SELECT
      disclosure_cycle,
      cdp_disclosing_org_number,
      row_order,
      'sectors_applied' AS col_name,
      TRIM(col5_Sectors_adaptation_action_applies_to) AS text_content
    FROM Q9_1_combined
    WHERE col5_Sectors_adaptation_action_applies_to IS NOT NULL AND TRIM(col5_Sectors_adaptation_action_applies_to)!= ''
    --and cdp_disclosing_org_number = 859097 -- Yamato
    UNION ALL
    SELECT
      disclosure_cycle,
      cdp_disclosing_org_number,
      row_order,
      'resilience_enhanced' AS col_name,
      TRIM(col6_Select_the_attributes_of_resilience_this_action_enhances) AS text_content
    FROM Q9_1_combined
    WHERE col6_Select_the_attributes_of_resilience_this_action_enhances IS NOT NULL AND TRIM(col6_Select_the_attributes_of_resilience_this_action_enhances)!= ''
    --and cdp_disclosing_org_number = 859097 -- Yamato
    UNION ALL
    SELECT
      disclosure_cycle,
      cdp_disclosing_org_number,
      row_order,
      'cobenefit_realized' AS col_name,
      TRIM(col7_Co_benefits_realized) AS text_content
    FROM Q9_1_combined
    WHERE col7_Co_benefits_realized IS NOT NULL AND TRIM(col7_Co_benefits_realized)!= ''
    --and cdp_disclosing_org_number = 859097 -- Yamato
    UNION ALL
    SELECT
      disclosure_cycle,
      cdp_disclosing_org_number,
      row_order,
      'timeframe' AS col_name,
      TRIM(col8_Timeframe_for_which_increased_resilience_is_expected_to_last) AS text_content
    FROM Q9_1_combined
    WHERE col8_Timeframe_for_which_increased_resilience_is_expected_to_last IS NOT NULL AND TRIM(col8_Timeframe_for_which_increased_resilience_is_expected_to_last)!= ''
    --and cdp_disclosing_org_number = 859097 -- Yamato
    UNION ALL
    SELECT
      disclosure_cycle,
      cdp_disclosing_org_number,
      row_order,
      'funding_source' AS col_name,
      TRIM(col11_Funding_source_s) AS text_content
    FROM Q9_1_combined
    WHERE col11_Funding_source_s IS NOT NULL AND TRIM(col11_Funding_source_s)!= ''
    --and cdp_disclosing_org_number = 859097 -- Yamato
    UNION ALL
    SELECT
      disclosure_cycle,
      cdp_disclosing_org_number,
      row_order,
      'action_status' AS col_name,
      TRIM(col12_Status_of_action_in_the_reporting_year) AS text_content
    FROM Q9_1_combined
    WHERE col12_Status_of_action_in_the_reporting_year IS NOT NULL AND TRIM(col12_Status_of_action_in_the_reporting_year)!= ''
    --and cdp_disclosing_org_number = 859097 -- Yamato
  ),
  -- Step 2: Detect language on unpivoted data
  DetectedLanguages AS (
    SELECT
      ut.disclosure_cycle,
      ut.cdp_disclosing_org_number,
      ut.row_order,
      ut.col_name,
      ut.text_content,
      -- Override per-field detection → 'en' for short, single-token, mostly-uppercase entries (MoSE, iOS, mRNA, etc.) 
      -- that the language detector misclassifies on too-short input
      CASE
        WHEN LENGTH(TRIM(ut.text_content)) <= 9
             AND ARRAY_LENGTH(SPLIT(TRIM(ut.text_content), ' ')) = 1
             AND NOT REGEXP_CONTAINS(ut.text_content, r'[^[:ascii:]]')
             AND LENGTH(REGEXP_REPLACE(ut.text_content, r'[^A-Z]', '')) * 2
                 > LENGTH(REGEXP_REPLACE(ut.text_content, r'[^A-Za-z]', ''))
          THEN 'en'
        ELSE JSON_VALUE(ml_translate_result, '$.languages[0].language_code')
      END AS detected_lang
    FROM ML.TRANSLATE(
            MODEL `{BQ_MODELS_DATASET}.translation_model`,
            TABLE UnpivotedTexts,
            STRUCT('detect_language' AS translate_mode)
         ) AS ut
    WHERE ml_translate_status = ''
  ),
  -- Step 3: Translate non-English texts
  Translations AS (
    SELECT
      disclosure_cycle,
      cdp_disclosing_org_number,
      row_order,
      col_name,
    --  ml_translate_result,
      JSON_VALUE(ml_translate_result, '$.translations[0].translated_text') AS translated_text
    FROM ML.TRANSLATE(
      MODEL `{BQ_MODELS_DATASET}.translation_model`,
      (SELECT disclosure_cycle, cdp_disclosing_org_number, row_order, col_name, protect_acronyms(text_content) AS text_content FROM DetectedLanguages WHERE detected_lang != 'en'),
      STRUCT('translate_text' AS translate_mode, 'en' AS target_language_code)
    )
    WHERE ml_translate_status = ''
  ),
  -- Step 4: Combine original and translated texts
  CombinedTranslations AS (
    SELECT
      dl.disclosure_cycle,
      dl.cdp_disclosing_org_number,
      dl.row_order,
      dl.col_name,
      IF(dl.detected_lang = 'en', dl.text_content, restore_acronyms(tr.translated_text, dl.text_content)) AS english_text
    FROM DetectedLanguages dl
    LEFT JOIN Translations tr ON dl.disclosure_cycle = tr.disclosure_cycle AND dl.cdp_disclosing_org_number = tr.cdp_disclosing_org_number AND dl.row_order = tr.row_order AND dl.col_name = tr.col_name
  ),
  -- Step 5a: Aggregate per row_order (raw, pre-format)
  PivotedTranslationsRaw AS (
    SELECT
      disclosure_cycle,
      cdp_disclosing_org_number,
      row_order,
      MAX(IF(col_name = 'action', english_text, NULL)) AS action_english,
      MAX(IF(col_name = 'hazard_addressed', english_text, NULL)) AS hazard_addressed_english_raw,
      MAX(IF(col_name = 'action_description', english_text, NULL)) AS action_description_english,
      MAX(IF(col_name = 'sectors_applied', english_text, NULL)) AS sectors_applied_english,
      MAX(IF(col_name = 'resilience_enhanced', english_text, NULL)) AS resilience_enhanced_english,
      MAX(IF(col_name = 'cobenefit_realized', english_text, NULL)) AS cobenefit_realized_english,
      MAX(IF(col_name = 'timeframe', english_text, NULL)) AS timeframe_english,
      MAX(IF(col_name = 'funding_source', english_text, NULL)) AS funding_source_english,
      MAX(IF(col_name = 'action_status', english_text, NULL)) AS action_status_english
    FROM CombinedTranslations
    GROUP BY disclosure_cycle,
      cdp_disclosing_org_number,
      row_order
  ),
  -- Step 5b: Apply format_hazard token-by-token to the pipe-joined hazard list
  PivotedTranslations AS (
    SELECT
      * EXCEPT(hazard_addressed_english_raw),
      CASE
        WHEN hazard_addressed_english_raw IS NULL THEN NULL
        ELSE (
          SELECT ARRAY_TO_STRING(ARRAY_AGG(format_hazard(h)), '|')
          FROM UNNEST(SPLIT(hazard_addressed_english_raw, '|')) AS h
        )
      END AS hazard_addressed_english
    FROM PivotedTranslationsRaw
  )
  -- Step 6: Append translated fields to the original hazard table
SELECT
  t.*,
  pt.action_english,
  pt.hazard_addressed_english,
  pt.action_description_english,
  pt.sectors_applied_english,
  pt.resilience_enhanced_english,
  pt.cobenefit_realized_english,
  pt.timeframe_english,
  pt.funding_source_english,
  pt.action_status_english
FROM Q9_1_combined t
LEFT JOIN PivotedTranslations pt
  ON t.disclosure_cycle = pt.disclosure_cycle
  AND t.cdp_disclosing_org_number = pt.cdp_disclosing_org_number
  AND t.row_order = pt.row_order
;
"""


In [52]:
query_job = bq_client.query(actions_translation_query)
query_job.result()
print(f"Successfully created {BQ_DATASET_PROCESSED}.action_translated")

Successfully created project-bb4fd058-24e7-4ccb-b06.CSTAR_2025_processed_v2.action_translated


In [53]:
fact_action_query = f"""
CREATE OR REPLACE TABLE `{BQ_DATASET_PROCESSED}.fact_action` AS
WITH
temp_fact_action as (
SELECT
    ht.disclosure_cycle,
    ht.cdp_disclosing_org_number,
    ht.disclosing_organization,
    ht.public_status,
    ht.action_english,
    STRING_AGG(distinct hazard_addressed_english, '|') AS hazard_addressed_english,
    STRING_AGG(distinct action_description_english, '|') AS action_description_english,
    STRING_AGG(distinct sectors_applied_english, '|') AS sectors_applied_english,
    STRING_AGG(distinct resilience_enhanced_english, '|') AS resilience_enhanced_english,
    STRING_AGG(distinct cobenefit_realized_english, '|') AS cobenefit_realized_english,
    -- logic for timeframe, use simple distinct aggregation
    STRING_AGG(distinct timeframe_english, '|') AS timeframe_english,
    STRING_AGG(distinct funding_source_english, '|') AS funding_source_english,
    STRING_AGG(distinct action_status_english, '|') AS action_status_english,
    SUM(col14_Total_cost_of_action_in_currency_specified_in_1_2 * dc.fx_rate) AS total_cost_usd
FROM
    `{BQ_DATASET_PROCESSED}.action_translated` ht
LEFT JOIN `{initial_central_dim_table_name}` dc
ON ht.cdp_disclosing_org_number = dc.cdp_disclosing_org_number
GROUP BY 1,2,3,4,5
ORDER BY 1,2,3,4,5
),

----- add disclosing_year and action index
ActionIndex AS
(
  SELECT
   action_english,
   ROW_NUMBER() OVER (ORDER BY action_english) AS action_index
  FROM temp_fact_action
  WHERE action_english IS NOT NULL
  AND TRIM(SPLIT(action_english, ':')[SAFE_OFFSET(0)]) != 'No adaptation action in place'
  GROUP BY 1
  ORDER BY 1
)
SELECT
   ht.disclosure_cycle,
   ht.cdp_disclosing_org_number,
   ht.disclosing_organization,
   ht.public_status,
   ht.action_english,
   (SELECT STRING_AGG(distinct TRIM(val), '|') FROM UNNEST(SPLIT(ht.hazard_addressed_english, '|')) val) AS hazard_addressed_english,
   action_description_english,
   (SELECT STRING_AGG(distinct TRIM(val), '|') FROM UNNEST(SPLIT(ht.sectors_applied_english, '|')) val) AS sectors_applied_english,
   (SELECT STRING_AGG(distinct TRIM(val), '|') FROM UNNEST(SPLIT(ht.resilience_enhanced_english, '|')) val) AS resilience_enhanced_english,
   (SELECT STRING_AGG(distinct TRIM(val), '|') FROM UNNEST(SPLIT(ht.cobenefit_realized_english, '|')) val) AS cobenefit_realized_english,
   -- logic for timeframe, use simple distinct aggregation
   (SELECT STRING_AGG(distinct TRIM(val), '|') FROM UNNEST(SPLIT(ht.timeframe_english, '|')) val) AS timeframe_english,
   (SELECT STRING_AGG(distinct TRIM(val), '|') FROM UNNEST(SPLIT(ht.funding_source_english, '|')) val) AS funding_source_english,
   (SELECT STRING_AGG(distinct TRIM(val), '|') FROM UNNEST(SPLIT(ht.action_status_english, '|')) val) AS action_status_english,
   total_cost_usd,
   ai.action_index,
   CAST(LEFT(ht.disclosure_cycle,4) AS INT) AS disclosing_year
FROM
    temp_fact_action ht
LEFT JOIN ActionIndex ai
ON ht.action_english = ai.action_english
WHERE ht.action_english IS NOT NULL
  AND TRIM(SPLIT(ht.action_english, ':')[SAFE_OFFSET(0)]) != 'No adaptation action in place';
"""

In [54]:
query_job = bq_client.query(fact_action_query)
query_job.result()
print(f"Successfully created {BQ_DATASET_PROCESSED}.fact_action")

Successfully created project-bb4fd058-24e7-4ccb-b06.CSTAR_2025_processed_v2.fact_action


In [55]:
# Ensure that `fact_action_final` table is named and accessible below and for migration to Cloud SQL.

fact_action_final_query = f"""
CREATE OR REPLACE TABLE `{BQ_DATASET_PROCESSED}.fact_action_final` AS
SELECT * FROM `{BQ_DATASET_PROCESSED}.fact_action`;
"""
bq_client.query(fact_action_final_query).result()
print(f"Created table {BQ_DATASET_PROCESSED}.fact_action_final → fact_action")


Created table project-bb4fd058-24e7-4ccb-b06.CSTAR_2025_processed_v2.fact_action_final → fact_action


### Fact Hazard

In [56]:
hazard_translation_query = rf"""
{ACRONYM_PROTECTION_SQL}
{HAZARD_FORMAT_SQL}
CREATE OR REPLACE TABLE `{BQ_DATASET_PROCESSED}.hazard_translated` AS
WITH
  -- Step 1: Unpivot columns for translation
  Q2_2_combined AS (
    SELECT * FROM `{BQ_DATASET_RAW}.Q2_2`
    FULL OUTER UNION ALL BY NAME
    SELECT * FROM `Missing_Data.quezon_Q2_2`
  ),
  UnpivotedTexts AS (
    SELECT
      disclosure_cycle,
      cdp_disclosing_org_number,
      row_order,
      'hazard' AS col_name,
      TRIM(col1_Climate_related_hazards) AS text_content
    FROM Q2_2_combined
    WHERE col1_Climate_related_hazards IS NOT NULL AND TRIM(col1_Climate_related_hazards) != ''
    --and cdp_disclosing_org_number = 859097 -- Yamato
    UNION ALL
    SELECT
      disclosure_cycle,
      cdp_disclosing_org_number,
      row_order,
      'population_exposed' AS col_name,
      TRIM(col2_Vulnerable_population_groups_most_exposed) AS text_content
    FROM Q2_2_combined
    WHERE col2_Vulnerable_population_groups_most_exposed IS NOT NULL AND TRIM(col2_Vulnerable_population_groups_most_exposed)!= ''
    --and cdp_disclosing_org_number =  859097
    UNION ALL
    SELECT
      disclosure_cycle,
      cdp_disclosing_org_number,
      row_order,
      'sectors_exposed' AS col_name,
      TRIM(col3_Sectors_most_exposed) AS text_content
    FROM Q2_2_combined
    WHERE col3_Sectors_most_exposed IS NOT NULL AND TRIM(col3_Sectors_most_exposed)!= ''
    --and cdp_disclosing_org_number =  859097
    UNION ALL
    SELECT
      disclosure_cycle,
      cdp_disclosing_org_number,
      row_order,
      'impacts' AS col_name,
      TRIM(col4_Describe_the_impacts_on_vulnerable_populations_and_sectors) AS text_content
    FROM Q2_2_combined
    WHERE col4_Describe_the_impacts_on_vulnerable_populations_and_sectors IS NOT NULL AND TRIM(col4_Describe_the_impacts_on_vulnerable_populations_and_sectors)!= ''
    --and cdp_disclosing_org_number =  859097
  ),
  -- Step 2: Detect language on unpivoted data
  DetectedLanguages AS (
    SELECT
      ut.disclosure_cycle,
      ut.cdp_disclosing_org_number,
      ut.row_order,
      ut.col_name,
      ut.text_content,
      -- Override per-field detection → 'en' for short, single-token, mostly-uppercase entries (MoSE, iOS, mRNA, etc.) 
      -- that the language detector misclassifies on too-short input
      CASE
        WHEN LENGTH(TRIM(ut.text_content)) <= 9
             AND ARRAY_LENGTH(SPLIT(TRIM(ut.text_content), ' ')) = 1
             AND NOT REGEXP_CONTAINS(ut.text_content, r'[^[:ascii:]]')
             AND LENGTH(REGEXP_REPLACE(ut.text_content, r'[^A-Z]', '')) * 2
                 > LENGTH(REGEXP_REPLACE(ut.text_content, r'[^A-Za-z]', ''))
          THEN 'en'
        ELSE JSON_VALUE(ml_translate_result, '$.languages[0].language_code')
      END AS detected_lang
    FROM ML.TRANSLATE(
            MODEL `{BQ_MODELS_DATASET}.translation_model`,
            TABLE UnpivotedTexts,
            STRUCT('detect_language' AS translate_mode)
         ) AS ut
    WHERE ml_translate_status = ''
  ),
  -- Step 3: Translate non-English texts
  Translations AS (
    SELECT
      disclosure_cycle,
      cdp_disclosing_org_number,
      row_order,
      col_name,
    --  ml_translate_result,
      JSON_VALUE(ml_translate_result, '$.translations[0].translated_text') AS translated_text
    FROM ML.TRANSLATE(
      MODEL `{BQ_MODELS_DATASET}.translation_model`,
      (SELECT disclosure_cycle, cdp_disclosing_org_number, row_order, col_name, protect_acronyms(text_content) AS text_content FROM DetectedLanguages WHERE detected_lang != 'en'),
      STRUCT('translate_text' AS translate_mode, 'en' AS target_language_code)
    )
    WHERE ml_translate_status = ''
  ),
  -- Step 4: Combine original and translated texts
  CombinedTranslations AS (
    SELECT
      dl.disclosure_cycle,
      dl.cdp_disclosing_org_number,
      dl.row_order,
      dl.col_name,
      IF(dl.detected_lang = 'en', dl.text_content, restore_acronyms(tr.translated_text, dl.text_content)) AS english_text
    FROM DetectedLanguages dl
    LEFT JOIN Translations tr ON dl.disclosure_cycle = tr.disclosure_cycle AND dl.cdp_disclosing_org_number = tr.cdp_disclosing_org_number AND dl.row_order = tr.row_order AND dl.col_name = tr.col_name
  ),
  -- Step 5: Pivot translations back into columns
  PivotedTranslations AS (
    SELECT
      disclosure_cycle,
      cdp_disclosing_org_number,
      row_order,
      format_hazard(MAX(IF(col_name = 'hazard', english_text, NULL))) AS hazard_english,
      MAX(IF(col_name = 'population_exposed', english_text, NULL)) AS population_exposed_english,
      MAX(IF(col_name = 'sectors_exposed', english_text, NULL)) AS sectors_exposed_english,
      MAX(IF(col_name = 'impacts', english_text, NULL)) AS impacts_english
    FROM CombinedTranslations
    GROUP BY disclosure_cycle,
      cdp_disclosing_org_number,
      row_order
  )
  -- Step 6: Append translated fields to the original hazard table
SELECT
  t.*,
  pt.hazard_english,
  pt.population_exposed_english,
  pt.sectors_exposed_english,
  pt.impacts_english
FROM Q2_2_combined t
LEFT JOIN PivotedTranslations pt
  ON t.disclosure_cycle = pt.disclosure_cycle
  AND t.cdp_disclosing_org_number = pt.cdp_disclosing_org_number
  AND t.row_order = pt.row_order
;
"""

In [58]:
query_job = bq_client.query(hazard_translation_query)
query_job.result()
print(f"Successfully created {BQ_DATASET_PROCESSED}.hazard_translated")

Successfully created project-bb4fd058-24e7-4ccb-b06.CSTAR_2025_processed_v2.hazard_translated


In [59]:
fact_hazard_query = f"""
-- Create Fact Hazard Table
CREATE OR REPLACE TABLE `{BQ_DATASET_PROCESSED}.fact_hazard` AS
WITH
 -- Step 1: create clean low and high for col5_Proportion_of_the_population_exposed_to_the_hazard
  population_pct AS (
    SELECT
        disclosure_cycle,
        cdp_disclosing_org_number,
        hazard_english,
        CASE
            WHEN col5_Proportion_of_the_population_exposed_to_the_hazard LIKE '%≤%' THEN 0  -- If it's ≤10, the floor is 0
            WHEN col5_Proportion_of_the_population_exposed_to_the_hazard is null or col5_Proportion_of_the_population_exposed_to_the_hazard ='Data is not available' THEN NULL
            ELSE CAST(REGEXP_EXTRACT(col5_Proportion_of_the_population_exposed_to_the_hazard, r'^\\d+') AS INT64)
        END AS clean_low,
        CASE
            WHEN col5_Proportion_of_the_population_exposed_to_the_hazard is null or col5_Proportion_of_the_population_exposed_to_the_hazard ='Data is not available' THEN NULL
            -- Extract the last number found in the string
            ELSE CAST(REGEXP_EXTRACT(col5_Proportion_of_the_population_exposed_to_the_hazard, r'(\\d+)\\s*%?$') AS INT64)
        END AS clean_high
    FROM `{BQ_DATASET_PROCESSED}.hazard_translated`
  )
-- Step 2: join with other aggregated fields to create initial table

  SELECT
    ht.disclosure_cycle,
    ht.cdp_disclosing_org_number,
    ht.disclosing_organization,
    ht.public_status,
    ht.hazard_english,
    STRING_AGG(distinct population_exposed_english, '|') AS population_exposed_english,
    STRING_AGG(distinct sectors_exposed_english, '|') AS sectors_exposed_english,
    STRING_AGG(distinct impacts_english, '|') AS impacts,
    -- Get the lowest starting number and the highest ending number
    CONCAT(MIN(pp.clean_low), '-', MAX(pp.clean_high), '%') AS population_range,
    MAX_BY(col6_Current_probability_of_hazard, CASE col6_Current_probability_of_hazard
        WHEN 'High'        THEN 6
        WHEN 'Medium High' THEN 5
        WHEN 'Medium'      THEN 4
        WHEN 'Medium Low'  THEN 3
        WHEN 'Low'         THEN 2
        WHEN 'Do not know' THEN 1
      ELSE 0
    END) AS hazard_probability,
    MAX_BY(col7_Current_magnitude_of_impact_of_hazard, CASE col7_Current_magnitude_of_impact_of_hazard
        WHEN 'High'        THEN 6
        WHEN 'Medium High' THEN 5
        WHEN 'Medium'      THEN 4
        WHEN 'Medium Low'  THEN 3
        WHEN 'Low'         THEN 2
        WHEN 'Do not know' THEN 1
      ELSE 0
    END) AS hazard_magnitude,
    MAX_BY(col8_Expected_future_change_in_hazard_intensity, CASE col8_Expected_future_change_in_hazard_intensity
        WHEN 'Increasing' THEN 3
        WHEN 'Decreasing' THEN 2
        WHEN 'Do not know' THEN 1
      ELSE 0
    END) AS intensity_change,
    MAX_BY(col9_Expected_future_change_in_hazard_frequency, CASE col9_Expected_future_change_in_hazard_frequency
        WHEN 'Increasing' THEN 3
        WHEN 'Decreasing' THEN 2
        WHEN 'Do not know' THEN 1
      ELSE 0
    END) AS frequency_change,
    STRING_AGG(distinct col10_Timeframe_of_expected_future_changes, '|') AS time_frame
FROM
    `{BQ_DATASET_PROCESSED}.hazard_translated` ht
  LEFT JOIN population_pct pp
   ON ht.disclosure_cycle = pp.disclosure_cycle
    AND ht.cdp_disclosing_org_number = pp.cdp_disclosing_org_number AND ht.hazard_english = pp.hazard_english
--WHERE  ht.cdp_disclosing_org_number = 50357
GROUP BY 1,2,3,4,5
ORDER BY 1,2,3,4,5

;
"""

In [60]:
query_job = bq_client.query(fact_hazard_query)
query_job.result()
print(f"Successfully created {BQ_DATASET_PROCESSED}.fact_hazard")

Successfully created project-bb4fd058-24e7-4ccb-b06.CSTAR_2025_processed_v2.fact_hazard


In [61]:
fact_hazard_AI_query = f"""
CREATE OR REPLACE TABLE `{BQ_DATASET_PROCESSED}.fact_hazard_AI` AS
SELECT
  ml_generate_text_result.disclosure_cycle,
  ml_generate_text_result.cdp_disclosing_org_number,
  ml_generate_text_result.public_status,
  ml_generate_text_result.hazard_english,
  ml_generate_text_result.population_exposed_english,
  ml_generate_text_result.sectors_exposed_english,
  ml_generate_text_result.impacts,
  ml_generate_text_result.population_range,
  ml_generate_text_result.hazard_probability,
  ml_generate_text_result.hazard_magnitude,
  ml_generate_text_result.intensity_change,
  ml_generate_text_result.frequency_change,
  ml_generate_text_result.time_frame,
  ml_generate_text_result.result AS summary_text
FROM
  AI.GENERATE_TEXT(
    MODEL `{BQ_MODELS_DATASET}.summary_model`,
    (
    SELECT
      fact_hazard.disclosure_cycle,
      fact_hazard.cdp_disclosing_org_number,
      fact_hazard.public_status,
      fact_hazard.hazard_english,
      fact_hazard.population_exposed_english,
      fact_hazard.sectors_exposed_english,
      fact_hazard.impacts,
      fact_hazard.population_range,
      fact_hazard.hazard_probability,
      fact_hazard.hazard_magnitude,
      fact_hazard.intensity_change,
      fact_hazard.frequency_change,
      fact_hazard.time_frame,
      CONCAT(
       'Summarize the climate hazard risk profile for ',
       fact_hazard.disclosing_organization,
       ' based on the following data. The summary should be a short paragraph around 25-50 words. \\n',
          CONCAT(
            'Hazard: ', IFNULL(fact_hazard.hazard_english, 'N/A'),
            ', Exposed Population: ', IFNULL(fact_hazard.population_exposed_english, 'N/A'),
            ', Exposed Sectors: ', IFNULL(fact_hazard.sectors_exposed_english, 'N/A'),
            ', Impact: ', IFNULL(fact_hazard.impacts, 'N/A'),
            ', % Exposed: ', IFNULL(fact_hazard.population_range, 'N/A'),
            ', Current Prob: ', IFNULL(fact_hazard.hazard_probability, 'N/A'),
            ', Current Mag: ', IFNULL(fact_hazard.hazard_magnitude, 'N/A'),
            ', Future Intensity: ', IFNULL(fact_hazard.intensity_change, 'N/A'),
            ', Future Freq: ', IFNULL(fact_hazard.frequency_change, 'N/A'),
            ', Timeframe: ', IFNULL(fact_hazard.time_frame, 'N/A')
           )
    ) AS prompt
FROM
  `{BQ_DATASET_PROCESSED}.fact_hazard` AS fact_hazard
  ),
    STRUCT(0.2 AS temperature)
    ) AS ml_generate_text_result
"""


In [62]:
query_job = bq_client.query(fact_hazard_AI_query)
query_job.result()
print(f"Successfully executed AI.GENERATE_TEXT query")

Successfully executed AI.GENERATE_TEXT query


In [63]:
fact_hazard_final_query = f"""
-- Step 4: add in hazard_rank
CREATE OR REPLACE TABLE `{BQ_DATASET_PROCESSED}.fact_hazard_final` AS
WITH
HazardScores AS (
  SELECT
    fact_hazard.disclosure_cycle,
    fact_hazard.cdp_disclosing_org_number,
    fact_hazard.hazard_english,
    CASE
      WHEN fact_hazard.hazard_probability = 'Low' THEN 1
      WHEN fact_hazard.hazard_probability = 'Medium Low' THEN 2
      WHEN fact_hazard.hazard_probability = 'Medium' THEN 3
      WHEN fact_hazard.hazard_probability = 'Medium High' THEN 4
      WHEN fact_hazard.hazard_probability = 'High' THEN 5
      ELSE NULL
  END
    AS probability_score,
    CASE
      WHEN fact_hazard.hazard_magnitude = 'Low' THEN 1
      WHEN fact_hazard.hazard_magnitude = 'Medium Low' THEN 2
      WHEN fact_hazard.hazard_magnitude = 'Medium' THEN 3
      WHEN fact_hazard.hazard_magnitude = 'Medium High' THEN 4
      WHEN fact_hazard.hazard_magnitude = 'High' THEN 5
      ELSE NULL
  END
    AS magnitude_score,
    CAST(REGEXP_EXTRACT(fact_hazard.population_range, r'(\\d+)\\s*%?$') AS INT64) AS population_score,
  FROM
    `{BQ_DATASET_PROCESSED}.fact_hazard` fact_hazard
),
HazardRanked AS (
  SELECT
    disclosure_cycle,
    cdp_disclosing_org_number,
    hazard_english,
    -- Calculate average score if probability and magnitude are available
    COALESCE( (probability_score + magnitude_score) / 2,
      -- Otherwise, use population exposed score
      population_score,
      -- If all are null, assign a very low score to be ranked last, or handle in the final step
      - 1  -- Assign a placeholder for null scores to ensure they are ranked last
      ) AS combined_score,
    ROW_NUMBER() OVER (PARTITION BY cdp_disclosing_org_number ORDER BY COALESCE( (probability_score + magnitude_score) / 2, population_score, -1) DESC,
      hazard_english  -- Tie-breaker for consistent results
      ) AS rn
  FROM
    HazardScores
)
SELECT
  A.*,
  HR.rn AS hazard_rank,
  CAST(LEFT(A.disclosure_cycle, 4) AS INT) AS disclosing_year
FROM `{BQ_DATASET_PROCESSED}.fact_hazard_AI` A
LEFT JOIN HazardRanked HR
  ON A.cdp_disclosing_org_number = HR.cdp_disclosing_org_number
  AND A.hazard_english = HR.hazard_english
;
"""

In [64]:
query_job = bq_client.query(fact_hazard_final_query)
query_job.result()
print(f"Successfully created {BQ_DATASET_PROCESSED}.fact_hazard_final")

Successfully created project-bb4fd058-24e7-4ccb-b06.CSTAR_2025_processed_v2.fact_hazard_final


### Fact Funding Gap

In [65]:
funding_translation_query = rf"""
{ACRONYM_PROTECTION_SQL}
CREATE OR REPLACE TABLE `{BQ_DATASET_PROCESSED}.funding_translated` AS
WITH
  -- Step 1: Unpivot columns for translation
  Q9_3_combined AS (
    SELECT * FROM `{BQ_DATASET_RAW}.Q9_3`
    FULL OUTER UNION ALL BY NAME
    SELECT * FROM `Missing_Data.quezon_Q9_3`
  ),
  UnpivotedTexts AS (
    SELECT
      disclosure_cycle,
      cdp_disclosing_org_number,
      row_order,
      'project_area' AS col_name,
      TRIM(col1_Project_area) AS text_content
    FROM Q9_3_combined
    WHERE col1_Project_area IS NOT NULL AND TRIM(col1_Project_area) != ''
   -- and cdp_disclosing_org_number = 31111 -- tokyo
    UNION ALL
    SELECT
      disclosure_cycle,
      cdp_disclosing_org_number,
      row_order,
      'project_title' AS col_name,
      TRIM(col2_Project_title) AS text_content
    FROM Q9_3_combined
    WHERE col2_Project_title IS NOT NULL AND TRIM(col2_Project_title)!= ''
   -- and cdp_disclosing_org_number =  31111
    UNION ALL
    SELECT
      disclosure_cycle,
      cdp_disclosing_org_number,
      row_order,
      'finance_status' AS col_name,
      TRIM(col4_Status_of_financing) AS text_content
    FROM Q9_3_combined
    WHERE col4_Status_of_financing IS NOT NULL AND TRIM(col4_Status_of_financing)!= ''
  --  and cdp_disclosing_org_number =  31111
    UNION ALL
    SELECT
      disclosure_cycle,
      cdp_disclosing_org_number,
      row_order,
      'finance_model' AS col_name,
      TRIM(col5_Identified_financing_model) AS text_content
    FROM Q9_3_combined
    WHERE col5_Identified_financing_model IS NOT NULL AND TRIM(col5_Identified_financing_model)!= ''
  --  and cdp_disclosing_org_number =  31111
    UNION ALL
    SELECT
      disclosure_cycle,
      cdp_disclosing_org_number,
      row_order,
      'project_descirption' AS col_name,
      TRIM(col6_Project_description_and_URL_link_if_applicable) AS text_content
    FROM Q9_3_combined
    WHERE col6_Project_description_and_URL_link_if_applicable IS NOT NULL AND TRIM(col6_Project_description_and_URL_link_if_applicable)!= ''
  --  and cdp_disclosing_org_number =  31111
  ),
  -- Step 2: Detect language on unpivoted data
  DetectedLanguages AS (
    SELECT
      ut.disclosure_cycle,
      ut.cdp_disclosing_org_number,
      ut.row_order,
      ut.col_name,
      ut.text_content,
      -- Override per-field detection → 'en' for short, single-token, mostly-uppercase entries (MoSE, iOS, mRNA, etc.) 
      -- that the language detector misclassifies on too-short input
      CASE
        WHEN LENGTH(TRIM(ut.text_content)) <= 9
             AND ARRAY_LENGTH(SPLIT(TRIM(ut.text_content), ' ')) = 1
             AND NOT REGEXP_CONTAINS(ut.text_content, r'[^[:ascii:]]')
             AND LENGTH(REGEXP_REPLACE(ut.text_content, r'[^A-Z]', '')) * 2
                 > LENGTH(REGEXP_REPLACE(ut.text_content, r'[^A-Za-z]', ''))
          THEN 'en'
        ELSE JSON_VALUE(ml_translate_result, '$.languages[0].language_code')
      END AS detected_lang
    FROM ML.TRANSLATE(
            MODEL `{BQ_MODELS_DATASET}.translation_model`,
            TABLE UnpivotedTexts,
            STRUCT('detect_language' AS translate_mode)
         ) AS ut
    WHERE ml_translate_status = ''
  ),
  -- Step 3: Translate non-English texts
  Translations AS (
    SELECT
      disclosure_cycle,
      cdp_disclosing_org_number,
      row_order,
      col_name,
    --  ml_translate_result,
      JSON_VALUE(ml_translate_result, '$.translations[0].translated_text') AS translated_text
    FROM ML.TRANSLATE(
      MODEL `{BQ_MODELS_DATASET}.translation_model`,
      (SELECT disclosure_cycle, cdp_disclosing_org_number, row_order, col_name, protect_acronyms(text_content) AS text_content FROM DetectedLanguages WHERE detected_lang != 'en'),
      STRUCT('translate_text' AS translate_mode, 'en' AS target_language_code)
    )
    WHERE ml_translate_status = ''
  ),
  -- Step 4: Combine original and translated texts
  CombinedTranslations AS (
    SELECT
      dl.disclosure_cycle,
      dl.cdp_disclosing_org_number,
      dl.row_order,
      dl.col_name,
      IF(dl.detected_lang = 'en', dl.text_content, restore_acronyms(tr.translated_text, dl.text_content)) AS english_text
    FROM DetectedLanguages dl
    LEFT JOIN Translations tr ON dl.disclosure_cycle = tr.disclosure_cycle AND dl.cdp_disclosing_org_number = tr.cdp_disclosing_org_number AND dl.row_order = tr.row_order AND dl.col_name = tr.col_name
  ),
  -- Step 5: Pivot translations back into columns
  PivotedTranslations AS (
    SELECT
      disclosure_cycle,
      cdp_disclosing_org_number,
      row_order,
      MAX(IF(col_name = 'project_area', english_text, NULL)) AS project_area_english,
      MAX(IF(col_name = 'project_title', english_text, NULL)) AS project_title_english,
      MAX(IF(col_name = 'finance_status', english_text, NULL)) AS finance_status_english,
      MAX(IF(col_name = 'finance_model', english_text, NULL)) AS finance_model_english,
      MAX(IF(col_name = 'project_descirption', english_text, NULL)) AS project_descirption_english
    FROM CombinedTranslations
    GROUP BY disclosure_cycle,
      cdp_disclosing_org_number,
      row_order
  )
  -- Step 6: Append translated fields to the original hazard table
SELECT
  t.*,
  pt.project_area_english,
  pt.project_title_english,
  pt.finance_status_english,
  pt.finance_model_english,
  pt.project_descirption_english
FROM Q9_3_combined t
LEFT JOIN PivotedTranslations pt
  ON t.disclosure_cycle = pt.disclosure_cycle
  AND t.cdp_disclosing_org_number = pt.cdp_disclosing_org_number
  AND t.row_order = pt.row_order
;"""

In [66]:
query_job = bq_client.query(funding_translation_query)
query_job.result()
print(f"Successfully created {BQ_DATASET_PROCESSED}.funding_translated")

Successfully created project-bb4fd058-24e7-4ccb-b06.CSTAR_2025_processed_v2.funding_translated


In [67]:
fact_funding_query = f"""
CREATE OR REPLACE TABLE `{BQ_DATASET_PROCESSED}.fact_funding_gap` AS
WITH InitialFunding AS (
SELECT
    ht.disclosure_cycle,
    ht.cdp_disclosing_org_number,
    ht.disclosing_organization,
    ht.public_status,
    ht.project_area_english,
    ht.project_title_english,
    STRING_AGG(distinct col3_Stage_of_project_development, '|') AS development_stage,
    STRING_AGG(distinct finance_status_english, '|') AS finance_status_english,
    STRING_AGG(distinct finance_model_english, '|') AS finance_model_english,
    STRING_AGG(distinct project_descirption_english, '|') AS project_descirption_english,
    SUM(col8_Total_cost_of_project_in_currency_specified_in_1_2 * dc.fx_rate) AS total_cost_usd,
    SUM(col9_Total_investment_cost_needed_if_relevant_in_currency_specified_in_1_2 * dc.fx_rate) AS total_needed_usd
FROM
    `{BQ_DATASET_PROCESSED}.funding_translated` ht
LEFT JOIN `{initial_central_dim_table_name}` dc
ON ht.cdp_disclosing_org_number = dc.cdp_disclosing_org_number
WHERE ht.project_area_english != 'No relevant projects'
  AND project_title_english is not null  AND project_title_english != ''
-- AND ht.cdp_disclosing_org_number = 55161
GROUP BY 1,2,3,4,5,6
ORDER BY 1,2,3,4,5,6
),
----- add disclosing_year and index
ProjectAreaIndex AS
(
  SELECT
    project_area_english,
    ROW_NUMBER() OVER (ORDER BY project_area_english) AS project_area_index
  FROM InitialFunding
  WHERE project_area_english IS NOT NULL
  AND project_area_english != 'No relevant projects'
  AND project_area_english != ''
  GROUP BY 1
)
SELECT
    ht.*,
    CAST(LEFT(ht.disclosure_cycle, 4) AS INT64) AS disclosing_year,
    pia.project_area_index,
    ROW_NUMBER() OVER (PARTITION BY ht.cdp_disclosing_org_number ORDER BY ht.project_title_english) AS project_index
FROM
    InitialFunding ht
LEFT JOIN ProjectAreaIndex pia ON ht.project_area_english = pia.project_area_english
WHERE ht.project_area_english IS NOT NULL
AND ht.project_title_english IS NOT NULL
AND ht.project_title_english != ''
AND ht.project_area_english != 'No relevant projects'
AND ht.project_area_english != ''
--AND ht.cdp_disclosing_org_number = 31181
; """

In [68]:
query_job = bq_client.query(fact_funding_query)
query_job.result()
print(f"Successfully created {BQ_DATASET_PROCESSED}.fact_funding_gap")

Successfully created project-bb4fd058-24e7-4ccb-b06.CSTAR_2025_processed_v2.fact_funding_gap


# Create final dim_central table

In [69]:
central_dimension_table_name_final = f"{BQ_DATASET_PROCESSED}.dim_cdp_geo_and_ecoregion"

part2 = f"""
-- add in geometry and ecoregion data
--Step 1: Convert JSON file to ndJSON using colab enterprise: https://pantheon.corp.google.com/vertex-ai/colab/overview?referrer=search&e=13802955&mods=-autopush_coliseum&project=project-bb4fd058-24e7-4ccb-b06&activeNb=projects%2Fproject-bb4fd058-24e7-4ccb-b06%2Flocations%2Fus-central1%2Frepositories%2F4f7cfb9a-cc3a-4d11-a321-90352d1e5ca3

--Step 2: load JSON file into BQ
LOAD DATA OVERWRITE `{BQ_DATASET_PROCESSED}.cities_raw`
(city_id STRING, geometry STRING)
FROM FILES (
  format = 'JSON',
  uris = ['{overture_cdp_geometry_json_file}']
);

--Step 3: join to get geometry and ecoregion for originally missing locations
create or replace table `{central_dimension_table_name_final}` as
(
  with
  -- Prepare Sector Data (`temp_sectors_processed`)
  temp_sectors_processed AS (
    SELECT
        cdp_disclosing_org_number,
        TRIM(sector_unnested) AS sector
    FROM
        `{BQ_DATASET_PROCESSED}.fact_hazard`,
        UNNEST(SPLIT(sectors_exposed_english, '|')) AS sector_unnested
    WHERE
        sectors_exposed_english IS NOT NULL
  ),

  -- Rank Sectors (`temp_ranked_sectors`)
  temp_ranked_sectors AS (
    SELECT
        cdp_disclosing_org_number,
        sector,
        COUNT(*) AS sector_frequency,
        ROW_NUMBER() OVER (PARTITION BY cdp_disclosing_org_number ORDER BY COUNT(*) DESC, sector ASC) AS sector_rank
    FROM
        temp_sectors_processed
    GROUP BY
        cdp_disclosing_org_number,
        sector
  ),

  -- Summarize Ranked Sectors (`temp_sector_summary`)
  temp_sector_summary AS (
    SELECT
        cdp_disclosing_org_number,
        STRING_AGG(sector, '|' ORDER BY sector_rank ASC) AS ranked_sectors
    FROM
        temp_ranked_sectors
    GROUP BY
        cdp_disclosing_org_number
  ),

  -- Summarize Ranked Hazards (`temp_hazard_summary`)
  temp_hazard_summary AS (
    SELECT
        cdp_disclosing_org_number,
        STRING_AGG(hazard_english, '|' ORDER BY hazard_rank ASC) AS ranked_hazards
    FROM
        `{BQ_DATASET_PROCESSED}.fact_hazard_final`
    GROUP BY
        cdp_disclosing_org_number
  ),

-- Create Updated Dimension Table (`dim_central_updated`)
templ_dim_table as (
SELECT
    dcb.*,
    tss.ranked_sectors,
    ths.ranked_hazards,
    ra.`Requesting Authorities` AS requesting_auth,
    ra.`Show Hazard Data_` AS ided_non_disclosers
FROM
    `{initial_central_dim_table_name}` AS dcb
LEFT JOIN
    temp_sector_summary AS tss
ON
    dcb.cdp_disclosing_org_number = tss.cdp_disclosing_org_number
LEFT JOIN
    temp_hazard_summary AS ths
ON
    dcb.cdp_disclosing_org_number = ths.cdp_disclosing_org_number
LEFT JOIN
    `{BQ_DATASET_RAW}.requesting_auth_non_disclosers` AS ra
ON
    dcb.cdp_disclosing_org_number = ra.`Org No_`
  ),

  overture as (
  SELECT *
  FROM `{BQ_DATASET_PROCESSED}.cities_raw`
  where geometry is not null
),

cdp as (
  select * FROM templ_dim_table
),

ecoregions as (
  select * from `project-bb4fd058-24e7-4ccb-b06.climate_zones.ecoregions`
),

cdp_with_overture as (
  -- Geometry sources, in priority order:
  --   1. Missing_Data.geometry-fixes (managed overlay — known geometry corrections)
  --   2. Missing_Data.missing-data   (managed overlay — geometry for orgs missing Overture coverage)
  --   3. Overture cities_raw         (default upstream geometry)
  --   4. dim_central.geometry         (fallback if nothing else available)
  select cdp.* EXCEPT(current_pop),
    COALESCE(SAFE_CAST(gf.population AS INT64), cdp.current_pop) AS current_pop,
    COALESCE(
      ST_GEOGFROMTEXT(
        REGEXP_REPLACE(gf.geometry_wkt, r' (Z|ZM|M) ', ' '),
        make_valid => TRUE
      ),
      ST_GEOGFROMTEXT(
        REGEXP_REPLACE(md.geometry_wkt, r' (Z|ZM|M) ', ' '),
        make_valid => TRUE
      ),
      SAFE.ST_GEOGFROMGEOJSON(overture.geometry, make_valid => TRUE)
    ) as geometry
  from cdp
  left join overture
    on cdp.cdp_disclosing_org_number = CAST(overture.city_id AS integer)
  left join `project-bb4fd058-24e7-4ccb-b06.Missing_Data.geometry-fixes` gf
    on cdp.cdp_disclosing_org_number = SAFE_CAST(gf.account_id AS INT64)
  left join `project-bb4fd058-24e7-4ccb-b06.Missing_Data.missing-data` md
    on cdp.cdp_disclosing_org_number = SAFE_CAST(md.account_number AS INT64)
),

cdp_with_ecoregion as (
  select cdp_with_overture.*, ecoregions.biome_name
  from cdp_with_overture
  left join ecoregions
  on ST_INTERSECTS(ecoregions.geometry, cdp_with_overture.geometry)
),

cdp_intersections AS (
  SELECT
    cdp.cdp_disclosing_org_number,
    eco.biome_name,
    -- Calculate the sum of intersection areas for each biome label
    SUM(ST_AREA(ST_INTERSECTION(cdp.geometry, eco.geometry))) AS total_intersection_area
  FROM cdp_with_overture AS cdp
  INNER JOIN ecoregions AS eco
    ON ST_INTERSECTS(cdp.geometry, eco.geometry)
  GROUP BY 1, 2
),

ranked_biomes AS (
  SELECT
    cdp_disclosing_org_number,
    biome_name,
    ROW_NUMBER() OVER(PARTITION BY cdp_disclosing_org_number ORDER BY total_intersection_area DESC) as rank
  FROM cdp_intersections
),

cdp_with_largest_area_ecoregion as (
SELECT
  cdp.*,
  CASE WHEN cdp.geometry IS NULL THEN 0 ELSE 1 END AS has_geometry,
  rb.biome_name as ecoregion
FROM cdp_with_overture AS cdp
LEFT JOIN ranked_biomes AS rb
  ON cdp.cdp_disclosing_org_number = rb.cdp_disclosing_org_number AND rb.rank = 1
)

select * from cdp_with_largest_area_ecoregion
);
"""

In [70]:
query_job = bq_client.query(part2)
query_job.result()
print(f"Successfully created {BQ_DATASET_PROCESSED}.dim_cdp_geo_and_ecoregion")

Successfully created project-bb4fd058-24e7-4ccb-b06.CSTAR_2025_processed_v2.dim_cdp_geo_and_ecoregion


# Solutions tables

#### Peer Solutions table

In [71]:
peer_list_query = f"""
{HAZARD_FORMAT_SQL}
CREATE OR REPLACE TABLE `{BQ_DATASET_PROCESSED}.peer_list_filtered` AS
-- assign ecoregion to locations
With
TargetCityTopHazards AS (
  -- Extract the top 4 hazards for each city
  SELECT
    h.cdp_disclosing_org_number AS target_org_id,
    h.ecoregion AS target_ecoregion,
    format_hazard(TRIM(hazard)) AS top_hazard
  FROM
    `{BQ_DATASET_PROCESSED}.dim_cdp_geo_and_ecoregion` AS h,
    UNNEST(SPLIT(h.ranked_hazards, '|')) AS hazard WITH OFFSET AS off
  WHERE
    off < 4 -- Corresponds to the first four positions (0, 1, 2, 3)
    AND ranked_hazards IS NOT NULL
    AND TRIM(hazard) != ''
    AND public_status = 'Public'
    AND h.ecoregion IS NOT NULL
),
PeerCityHazardsAddressed AS (
  -- Get all unique hazards addressed by each city's actions
  SELECT DISTINCT
    a.cdp_disclosing_org_number AS peer_org_id,
    b.ecoregion AS peer_ecoregion,
    a.action,
    format_hazard(TRIM(a.hazard_addressed)) AS hazard_addressed
  FROM
    (SELECT
       cdp_disclosing_org_number,
       col2_Action AS action,
       t0 AS hazard_addressed
    FROM `{BQ_DATASET_RAW}.Q9_1` AS Q9_1,
    UNNEST(SPLIT(Q9_1.col3_Climate_hazard_s_that_action_addresses, '|')) AS t0
    ) AS a
  INNER JOIN `{BQ_DATASET_PROCESSED}.dim_cdp_geo_and_ecoregion` AS b
   ON a.cdp_disclosing_org_number = b.cdp_disclosing_org_number
  WHERE TRIM(a.hazard_addressed)!= 'Action does not address hazard'
    AND TRIM(a.hazard_addressed) != ''
    AND TRIM(a.hazard_addressed) IS NOT NULL
    AND NOT REGEXP_CONTAINS(
    LOWER(TRIM(a.hazard_addressed)),
    r'\bother:')
    AND b.public_status = 'Public'
    AND b.ecoregion IS NOT NULL
    AND TRIM(a.action) != ''
  AND TRIM(SPLIT(a.action, ':')[SAFE_OFFSET(0)]) != 'No adaptation action in place'
  AND NOT REGEXP_CONTAINS(
    LOWER(TRIM(SPLIT(a.action, ':')[SAFE_OFFSET(0)])),
    r'\bother\b' )
  AND NOT REGEXP_CONTAINS(
    LOWER(TRIM(SPLIT(a.action, ':')[SAFE_OFFSET(1)])),
    r'\bother\b')
),
PeerList AS (
-- Find list of peer city actions addresses one of the target city's top hazards
SELECT DISTINCT
  TCTH.target_org_id,
  TCTH.top_hazard,
  PCHA.peer_org_id,
  PCHA.action
FROM
  TargetCityTopHazards AS TCTH
INNER JOIN
  PeerCityHazardsAddressed AS PCHA
  ON TCTH.target_ecoregion = PCHA.peer_ecoregion -- same ecoregions
  AND TCTH.top_hazard = PCHA.hazard_addressed
WHERE
  TCTH.target_org_id != PCHA.peer_org_id -- Assuming a city cannot be its own peer
ORDER BY
  1,2,3,4
),
PeerActions AS (
  SELECT
    pc.target_org_id,
    pc.top_hazard,
    pc.peer_org_id,
    q9.action_english,
    q9.action_index,
    TRIM(SPLIT(q9.action_english, ':')[SAFE_OFFSET(0)]) AS solution_category,
    TRIM(SPLIT(q9.action_english, ':')[SAFE_OFFSET(1)]) AS solution,
    q9.hazard_addressed_english as hazard_addressed_english
  FROM PeerList AS pc
  INNER JOIN `{BQ_DATASET_PROCESSED}.fact_action` q9
  ON pc.peer_org_id = q9.cdp_disclosing_org_number AND pc.action = q9.action_english
 WHERE TRIM(q9.action_english) != ''
  AND TRIM(SPLIT(q9.action_english, ':')[SAFE_OFFSET(0)]) != 'No adaptation action in place'
  AND NOT REGEXP_CONTAINS(
    LOWER(TRIM(SPLIT(q9.action_english, ':')[SAFE_OFFSET(0)])),
    r'\bother\b'
  )
  AND NOT REGEXP_CONTAINS(
    LOWER(TRIM(SPLIT(q9.action_english, ':')[SAFE_OFFSET(1)])),
    r'\bother\b'
  )
 --and target_org_id=1093
),
-- aggregate peer actions
AggPeerActions AS (
  -- Step 1: Get your accurate counts first
  SELECT
    target_org_id,
    top_hazard,
    solution_category,
    solution,
    action_english,
    action_index,
    COUNT(*) AS action_count,
    COUNT(DISTINCT peer_org_id) AS org_count
  FROM PeerActions
  GROUP BY 1,2,3,4,5,6
),
RankAction AS (
SELECT
    target_org_id,
    top_hazard,
    solution_category,
    solution,
    action_english,
    action_index,
    action_count,
    ROW_NUMBER() OVER (
      PARTITION BY target_org_id, top_hazard, solution_category
      ORDER BY action_count DESC, org_count DESC
    ) AS action_rank
  FROM AggPeerActions
  WHERE top_hazard IS NOT NULL
  AND top_hazard!=''
),
HazardLevel AS (
SELECT DISTINCT
  a.target_org_id,
  a.top_hazard,
  a.peer_org_id,
  a.action_english
FROM PeerActions a
INNER JOIN RankAction b
ON a.target_org_id = b.target_org_id
AND a.top_hazard = b.top_hazard
AND a.action_english = b.action_english
WHERE action_rank <= 5
),
AllLevel AS (
SELECT DISTINCT
  a.target_org_id,
  'All' AS top_hazard,
  a.peer_org_id,
  a.action_english
FROM PeerActions a
INNER JOIN RankAction b
  ON a.target_org_id = b.target_org_id
  AND a.action_index = b.action_index
WHERE b.action_rank <= 5
)
SELECT
*
FROM HazardLevel
UNION ALL
SELECT
*
FROM ALLLevel
; """

In [72]:
query_job = bq_client.query(peer_list_query)
query_job.result()
print(f"Successfully created {BQ_DATASET_PROCESSED}.peer_list_filtered")

Successfully created project-bb4fd058-24e7-4ccb-b06.CSTAR_2025_processed_v2.peer_list_filtered


In [73]:
peer_solutions_query = f"""
CREATE OR REPLACE TABLE `{BQ_DATASET_PROCESSED}.peer_solutions` AS
WITH
PeerActions AS (
  SELECT
    pc.target_org_id,
    pc.top_hazard,
    pc.peer_org_id,
    q9.action_english,
    q9.action_index,
    TRIM(SPLIT(q9.action_english, ':')[SAFE_OFFSET(0)]) AS solution_category,
    TRIM(SPLIT(q9.action_english, ':')[SAFE_OFFSET(1)]) AS solution,
    q9.hazard_addressed_english as hazard_addressed_english
  FROM `{BQ_DATASET_PROCESSED}.peer_list_filtered` AS pc
  INNER JOIN `{BQ_DATASET_PROCESSED}.fact_action` q9
  ON pc.peer_org_id = q9.cdp_disclosing_org_number AND pc.action_english = q9.action_english
 WHERE TRIM(q9.action_english) != ''
  AND TRIM(SPLIT(q9.action_english, ':')[SAFE_OFFSET(0)]) != 'No adaptation action in place'
  AND NOT REGEXP_CONTAINS(
    LOWER(TRIM(SPLIT(q9.action_english, ':')[SAFE_OFFSET(0)])),
    r'\bother\b'
  )
  AND NOT REGEXP_CONTAINS(
    LOWER(TRIM(SPLIT(q9.action_english, ':')[SAFE_OFFSET(1)])),
    r'\bother\b'
  )
),
-- Model 2: action_count = distinct peers in target's full pool who use this action
--          (regardless of which top-4 hazard they address with it).
ActionTotals AS (
  SELECT
    target_org_id,
    solution_category,
    solution,
    action_english,
    action_index,
    COUNT(DISTINCT peer_org_id) AS action_count,
    COUNT(DISTINCT peer_org_id) AS org_count -- now the same as action_count, left to minimize code changes
  FROM PeerActions
  WHERE top_hazard IS NOT NULL AND top_hazard != ''
  GROUP BY 1, 2, 3, 4, 5
),
-- For per-hazard views: which actions are addressed by at least one peer for this hazard
HazardActions AS (
  SELECT DISTINCT
    pa.target_org_id,
    pa.top_hazard,
    pa.solution_category,
    pa.solution,
    pa.action_english,
    pa.action_index
  FROM PeerActions pa
  WHERE pa.top_hazard IS NOT NULL AND pa.top_hazard != ''
),
-- Per-hazard ranking (action_count comes from ActionTotals — broader pool count)
HazardRanked AS (
  SELECT
    ha.target_org_id,
    ha.top_hazard,
    ha.solution_category,
    ha.solution,
    ha.action_english,
    ha.action_index,
    tot.action_count,
    tot.org_count,
    ROW_NUMBER() OVER (
      PARTITION BY ha.target_org_id, ha.top_hazard, ha.solution_category
      ORDER BY tot.action_count DESC, tot.org_count DESC, ha.solution_category, ha.solution
    ) AS action_rank
  FROM HazardActions ha
  INNER JOIN ActionTotals tot
    ON ha.target_org_id = tot.target_org_id
   AND ha.action_english = tot.action_english
),
-- 5/category cap, then 25-overall cap per (target, hazard)
HazardCapped AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY target_org_id, top_hazard
      ORDER BY action_count DESC, org_count DESC, solution_category, solution
    ) AS overall_rank
  FROM HazardRanked
  WHERE action_rank <= 5
),
-- Pool size per target = peers in pool (across all top-4 hazards), shared across views
PoolSizePerTarget AS (
  SELECT target_org_id, COUNT(DISTINCT peer_org_id) AS peer_org_cnt
  FROM PeerActions
  GROUP BY 1
),
HazardLevel AS (
  SELECT
    2025 as disclosing_year,
    HC.target_org_id,
    HC.top_hazard AS hazard_filter,
    HC.solution_category,
    HC.solution,
    HC.overall_rank AS action_rank,
    HC.action_english,
    HC.action_index,
    HC.top_hazard AS hazard_addressed,
    PT.peer_org_cnt,
    HC.action_count,
    SAFE_DIVIDE(HC.action_count, PT.peer_org_cnt) AS pct_peers
  FROM HazardCapped HC
  INNER JOIN PoolSizePerTarget PT USING (target_org_id)
  WHERE HC.overall_rank <= 25
),
-- 'All' view: rank actions across hazards, apply same 5/cat + 25-overall
AllRanked AS (
  SELECT
    tot.*,
    ROW_NUMBER() OVER (
      PARTITION BY tot.target_org_id, tot.solution_category
      ORDER BY tot.action_count DESC, tot.org_count DESC, tot.solution_category, tot.solution
    ) AS action_rank
  FROM ActionTotals tot
),
AllCapped AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY target_org_id
      ORDER BY action_count DESC, org_count DESC, solution_category, solution
    ) AS overall_rank
  FROM AllRanked
  WHERE action_rank <= 5
),
HazardsByAction AS (
  SELECT target_org_id, action_index,
    STRING_AGG(DISTINCT top_hazard, ' | ') AS hazards_addressed
  FROM PeerActions
  WHERE top_hazard IS NOT NULL AND top_hazard != ''
  GROUP BY 1, 2
),
PeerSolutions AS (
  SELECT * FROM HazardLevel
  UNION ALL
  SELECT
    2025 AS disclosing_year,
    AC.target_org_id,
    'All' AS hazard_filter,
    AC.solution_category,
    AC.solution,
    AC.overall_rank AS action_rank,
    AC.action_english,
    AC.action_index,
    HBA.hazards_addressed AS hazard_addressed,
    PT.peer_org_cnt,
    AC.action_count,
    SAFE_DIVIDE(AC.action_count, PT.peer_org_cnt) AS pct_peers
  FROM AllCapped AC
  INNER JOIN PoolSizePerTarget PT USING (target_org_id)
  LEFT JOIN HazardsByAction HBA USING (target_org_id, action_index)
  WHERE AC.overall_rank <= 25
)
SELECT
  a.*,
  CASE WHEN b.action_index IS NOT NULL THEN TRUE ELSE FALSE END AS has_local_action
FROM PeerSolutions a
LEFT JOIN `{BQ_DATASET_PROCESSED}`.`fact_action_final` b
ON a.target_org_id = b.cdp_disclosing_org_number
AND a.action_index = b.action_index
;"""

In [74]:
query_job = bq_client.query(peer_solutions_query)
query_job.result()
print(f"Successfully created {BQ_DATASET_PROCESSED}.peer_solutions")

Successfully created project-bb4fd058-24e7-4ccb-b06.CSTAR_2025_processed_v2.peer_solutions


### Solution Examples

In [75]:
solution_examples_query = f"""
CREATE OR REPLACE TABLE `{BQ_DATASET_PROCESSED}.solution_examples` AS
WITH LessThan10 AS (
SELECT
  2025 as disclosing_year,
  ps.target_org_id,
  ps.hazard_filter,
  ps.action_english,
  p.peer_org_id,
  a.disclosing_organization as peer_org_name,
  a.action_index,
  a.hazard_addressed_english,
  a.action_description_english,
  a.sectors_applied_english,
  a.resilience_enhanced_english,
  a.cobenefit_realized_english,
  a.timeframe_english,
  a.funding_source_english,
  a.action_status_english,
  a.total_cost_usd,
  1 as completeness_score
FROM `{BQ_DATASET_PROCESSED}.peer_solutions` ps
inner join `{BQ_DATASET_PROCESSED}.fact_action` a
on ps.action_english = a.action_english
inner join`{BQ_DATASET_PROCESSED}.peer_list_filtered` p
on a.cdp_disclosing_org_number = p.peer_org_id and p.target_org_id = ps.target_org_id and p.action_english = ps.action_english and p.top_hazard = ps.hazard_filter
WHERE action_count <= 10
--and ps.target_org_id = 1093
--and ps.action_english ='Economic actions: Payments for ecosystem services'
group by 1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
),
MoreThan10 AS (
SELECT
  2025 as disclosing_year,
  ps.target_org_id,
  ps.hazard_filter,
  ps.action_english,
  p.peer_org_id,
  a.disclosing_organization as peer_org_name,
  a.action_index,
  a.hazard_addressed_english,
  a.action_description_english,
  a.sectors_applied_english,
  a.resilience_enhanced_english,
  a.cobenefit_realized_english,
  a.timeframe_english,
  a.funding_source_english,
  a.action_status_english,
  a.total_cost_usd,
  -- Calculate how many fields are filled (not null)
  (CASE WHEN a.hazard_addressed_english IS NOT NULL THEN 1 ELSE 0 END +
   CASE WHEN a.action_description_english IS NOT NULL THEN 10 ELSE 0 END +
   CASE WHEN a.sectors_applied_english IS NOT NULL THEN 1 ELSE 0 END +
   CASE WHEN a.resilience_enhanced_english IS NOT NULL THEN 1 ELSE 0 END +
   CASE WHEN a.cobenefit_realized_english IS NOT NULL THEN 1 ELSE 0 END +
   CASE WHEN a.timeframe_english IS NOT NULL THEN 1 ELSE 0 END +
   CASE WHEN a.funding_source_english IS NOT NULL THEN 1 ELSE 0 END +
   CASE WHEN a.action_status_english IS NOT NULL THEN 1 ELSE 0 END +
   CASE WHEN a.total_cost_usd IS NOT NULL THEN 1 ELSE 0 END) AS completeness_score
FROM `{BQ_DATASET_PROCESSED}.peer_solutions` ps
inner join `{BQ_DATASET_PROCESSED}.fact_action` a
on ps.action_english = a.action_english
inner join`{BQ_DATASET_PROCESSED}.peer_list_filtered` p
on a.cdp_disclosing_org_number = p.peer_org_id and p.target_org_id = ps.target_org_id and p.action_english = ps.action_english and p.top_hazard = ps.hazard_filter
WHERE action_count > 10
--and ps.target_org_id = 1093
--and ps.action_english ='Educational/Informational actions: Early warning and response systems'
group by 1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
)
SELECT
 *
FROM LessThan10
UNION ALL
(
  SELECT
    disclosing_year,
    target_org_id,
    hazard_filter,
    action_english,
    peer_org_id,
    peer_org_name,
    action_index,
    hazard_addressed_english,
    action_description_english,
    sectors_applied_english,
    resilience_enhanced_english,
    cobenefit_realized_english,
    timeframe_english,
    funding_source_english,
    action_status_english,
    total_cost_usd,
    completeness_score
  FROM (
         SELECT
         *,
         ROW_NUMBER() OVER (PARTITION BY target_org_id, hazard_filter, action_english ORDER BY completeness_score DESC) AS completeness_rank
         FROM MoreThan10 )
  WHERE completeness_rank <= 10
)
; """

In [76]:
query_job = bq_client.query(solution_examples_query)
query_job.result()
print(f"Successfully created {BQ_DATASET_PROCESSED}.solution_examples")

Successfully created project-bb4fd058-24e7-4ccb-b06.CSTAR_2025_processed_v2.solution_examples


In [77]:
# The Cloud SQL migrate script reads `fact_goal_final`, `fact_funding_gap_final`, and `peer_solutions_final`, so rename to final aliases.

rename_tables = [
    ('fact_goal_final',         'fact_goal'),
    ('fact_funding_gap_final',  'fact_funding_gap'),
    ('peer_solutions_final',    'peer_solutions'),
]
for final_name, base_name in rename_tables:
    q = f"""
    CREATE OR REPLACE TABLE `{BQ_DATASET_PROCESSED}.{final_name}` AS
    SELECT * FROM `{BQ_DATASET_PROCESSED}.{base_name}`;
    """
    bq_client.query(q).result()
    print(f"Created table {BQ_DATASET_PROCESSED}.{final_name} → {base_name}")


Created table project-bb4fd058-24e7-4ccb-b06.CSTAR_2025_processed_v2.fact_goal_final → fact_goal
Created table project-bb4fd058-24e7-4ccb-b06.CSTAR_2025_processed_v2.fact_funding_gap_final → fact_funding_gap
Created table project-bb4fd058-24e7-4ccb-b06.CSTAR_2025_processed_v2.peer_solutions_final → peer_solutions


#### Translation audit table

Per-cell audit of routing + option-3 heuristic triggers + acronym-regex coverage. Covers `action` / `hazard` / `funding` pipelines (those with intermediate `*_translated` tables). The goal pipeline aggregates inline into `fact_goal` (no `goal_translated` intermediate), so goal audit needs separate spot checks against `fact_goal` directly.

Columns produced:
- `triggered_heuristic` — the option-3 short-token-uppercase heuristic from `DetectedLanguages` would have matched this raw text
- `has_acronym_in_raw`, `acronyms_in_raw` — whether the protect_acronyms regex catches anything in the raw text + the list of detected acronyms
- `route` — `kept_verbatim` / `translated_changed` / `output_null` / `output_empty`
- `raw_has_non_ascii`, `is_single_token`, `length_chars` — additional signals

Useful spot-check queries:
```sql
-- Every row that hit the new heuristic (option 3): MoSE / iOS / mRNA / etc.
SELECT * FROM translation_audit WHERE triggered_heuristic;

-- Safety: did the heuristic over-fire on real foreign content?
SELECT * FROM translation_audit WHERE triggered_heuristic AND raw_has_non_ascii;

-- Acronyms that may not have round-tripped (regex caught them in raw, but english differs)
SELECT * FROM translation_audit
WHERE has_acronym_in_raw AND NOT triggered_heuristic
  AND english_text IS NOT NULL AND english_text != raw_text;
```

In [78]:
# Translation audit: per-cell record of routing + option-3 heuristic triggers + acronym presence.

translation_audit_query = f"""
CREATE OR REPLACE TABLE `{BQ_DATASET_PROCESSED}.translation_audit` AS
WITH raw_unpivoted AS (
  -- ACTION: Q9_1 (9 columns)
  SELECT 'action' AS pipeline, 'action' AS col_name, cdp_disclosing_org_number, row_order, TRIM(col2_Action) AS raw_text
    FROM `{BQ_DATASET_RAW}.Q9_1` WHERE col2_Action IS NOT NULL AND TRIM(col2_Action) != ''
  UNION ALL SELECT 'action','hazard_addressed',     cdp_disclosing_org_number, row_order, TRIM(col3_Climate_hazard_s_that_action_addresses)
    FROM `{BQ_DATASET_RAW}.Q9_1` WHERE col3_Climate_hazard_s_that_action_addresses IS NOT NULL AND TRIM(col3_Climate_hazard_s_that_action_addresses) != ''
  UNION ALL SELECT 'action','action_description',   cdp_disclosing_org_number, row_order, TRIM(col4_Action_description_and_web_link_to_further_information)
    FROM `{BQ_DATASET_RAW}.Q9_1` WHERE col4_Action_description_and_web_link_to_further_information IS NOT NULL AND TRIM(col4_Action_description_and_web_link_to_further_information) != ''
  UNION ALL SELECT 'action','sectors_applied',      cdp_disclosing_org_number, row_order, TRIM(col5_Sectors_adaptation_action_applies_to)
    FROM `{BQ_DATASET_RAW}.Q9_1` WHERE col5_Sectors_adaptation_action_applies_to IS NOT NULL AND TRIM(col5_Sectors_adaptation_action_applies_to) != ''
  UNION ALL SELECT 'action','resilience_enhanced',  cdp_disclosing_org_number, row_order, TRIM(col6_Select_the_attributes_of_resilience_this_action_enhances)
    FROM `{BQ_DATASET_RAW}.Q9_1` WHERE col6_Select_the_attributes_of_resilience_this_action_enhances IS NOT NULL AND TRIM(col6_Select_the_attributes_of_resilience_this_action_enhances) != ''
  UNION ALL SELECT 'action','cobenefit_realized',   cdp_disclosing_org_number, row_order, TRIM(col7_Co_benefits_realized)
    FROM `{BQ_DATASET_RAW}.Q9_1` WHERE col7_Co_benefits_realized IS NOT NULL AND TRIM(col7_Co_benefits_realized) != ''
  UNION ALL SELECT 'action','timeframe',            cdp_disclosing_org_number, row_order, TRIM(col8_Timeframe_for_which_increased_resilience_is_expected_to_last)
    FROM `{BQ_DATASET_RAW}.Q9_1` WHERE col8_Timeframe_for_which_increased_resilience_is_expected_to_last IS NOT NULL AND TRIM(col8_Timeframe_for_which_increased_resilience_is_expected_to_last) != ''
  UNION ALL SELECT 'action','funding_source',       cdp_disclosing_org_number, row_order, TRIM(col11_Funding_source_s)
    FROM `{BQ_DATASET_RAW}.Q9_1` WHERE col11_Funding_source_s IS NOT NULL AND TRIM(col11_Funding_source_s) != ''
  UNION ALL SELECT 'action','action_status',        cdp_disclosing_org_number, row_order, TRIM(col12_Status_of_action_in_the_reporting_year)
    FROM `{BQ_DATASET_RAW}.Q9_1` WHERE col12_Status_of_action_in_the_reporting_year IS NOT NULL AND TRIM(col12_Status_of_action_in_the_reporting_year) != ''
  -- HAZARD: Q2_2 (4 columns)
  UNION ALL SELECT 'hazard','hazard',               cdp_disclosing_org_number, row_order, TRIM(col1_Climate_related_hazards)
    FROM `{BQ_DATASET_RAW}.Q2_2` WHERE col1_Climate_related_hazards IS NOT NULL AND TRIM(col1_Climate_related_hazards) != ''
  UNION ALL SELECT 'hazard','population_exposed',   cdp_disclosing_org_number, row_order, TRIM(col2_Vulnerable_population_groups_most_exposed)
    FROM `{BQ_DATASET_RAW}.Q2_2` WHERE col2_Vulnerable_population_groups_most_exposed IS NOT NULL AND TRIM(col2_Vulnerable_population_groups_most_exposed) != ''
  UNION ALL SELECT 'hazard','sectors_exposed',      cdp_disclosing_org_number, row_order, TRIM(col3_Sectors_most_exposed)
    FROM `{BQ_DATASET_RAW}.Q2_2` WHERE col3_Sectors_most_exposed IS NOT NULL AND TRIM(col3_Sectors_most_exposed) != ''
  UNION ALL SELECT 'hazard','impacts',              cdp_disclosing_org_number, row_order, TRIM(col4_Describe_the_impacts_on_vulnerable_populations_and_sectors)
    FROM `{BQ_DATASET_RAW}.Q2_2` WHERE col4_Describe_the_impacts_on_vulnerable_populations_and_sectors IS NOT NULL AND TRIM(col4_Describe_the_impacts_on_vulnerable_populations_and_sectors) != ''
  -- FUNDING: Q9_3 (5 columns)
  UNION ALL SELECT 'funding','project_area',        cdp_disclosing_org_number, row_order, TRIM(col1_Project_area)
    FROM `{BQ_DATASET_RAW}.Q9_3` WHERE col1_Project_area IS NOT NULL AND TRIM(col1_Project_area) != ''
  UNION ALL SELECT 'funding','project_title',       cdp_disclosing_org_number, row_order, TRIM(col2_Project_title)
    FROM `{BQ_DATASET_RAW}.Q9_3` WHERE col2_Project_title IS NOT NULL AND TRIM(col2_Project_title) != ''
  UNION ALL SELECT 'funding','finance_status',      cdp_disclosing_org_number, row_order, TRIM(col4_Status_of_financing)
    FROM `{BQ_DATASET_RAW}.Q9_3` WHERE col4_Status_of_financing IS NOT NULL AND TRIM(col4_Status_of_financing) != ''
  UNION ALL SELECT 'funding','finance_model',       cdp_disclosing_org_number, row_order, TRIM(col5_Identified_financing_model)
    FROM `{BQ_DATASET_RAW}.Q9_3` WHERE col5_Identified_financing_model IS NOT NULL AND TRIM(col5_Identified_financing_model) != ''
  UNION ALL SELECT 'funding','project_descirption', cdp_disclosing_org_number, row_order, TRIM(col6_Project_description_and_URL_link_if_applicable)
    FROM `{BQ_DATASET_RAW}.Q9_3` WHERE col6_Project_description_and_URL_link_if_applicable IS NOT NULL AND TRIM(col6_Project_description_and_URL_link_if_applicable) != ''
),
translated_unpivoted AS (
  SELECT 'action' AS pipeline,'action'              AS col_name, cdp_disclosing_org_number, row_order, action_english              AS english_text FROM `{BQ_DATASET_PROCESSED}.action_translated`
  UNION ALL SELECT 'action','hazard_addressed',     cdp_disclosing_org_number, row_order, hazard_addressed_english     FROM `{BQ_DATASET_PROCESSED}.action_translated`
  UNION ALL SELECT 'action','action_description',   cdp_disclosing_org_number, row_order, action_description_english   FROM `{BQ_DATASET_PROCESSED}.action_translated`
  UNION ALL SELECT 'action','sectors_applied',      cdp_disclosing_org_number, row_order, sectors_applied_english      FROM `{BQ_DATASET_PROCESSED}.action_translated`
  UNION ALL SELECT 'action','resilience_enhanced',  cdp_disclosing_org_number, row_order, resilience_enhanced_english  FROM `{BQ_DATASET_PROCESSED}.action_translated`
  UNION ALL SELECT 'action','cobenefit_realized',   cdp_disclosing_org_number, row_order, cobenefit_realized_english   FROM `{BQ_DATASET_PROCESSED}.action_translated`
  UNION ALL SELECT 'action','timeframe',            cdp_disclosing_org_number, row_order, timeframe_english            FROM `{BQ_DATASET_PROCESSED}.action_translated`
  UNION ALL SELECT 'action','funding_source',       cdp_disclosing_org_number, row_order, funding_source_english       FROM `{BQ_DATASET_PROCESSED}.action_translated`
  UNION ALL SELECT 'action','action_status',        cdp_disclosing_org_number, row_order, action_status_english        FROM `{BQ_DATASET_PROCESSED}.action_translated`
  UNION ALL SELECT 'hazard','hazard',               cdp_disclosing_org_number, row_order, hazard_english               FROM `{BQ_DATASET_PROCESSED}.hazard_translated`
  UNION ALL SELECT 'hazard','population_exposed',   cdp_disclosing_org_number, row_order, population_exposed_english   FROM `{BQ_DATASET_PROCESSED}.hazard_translated`
  UNION ALL SELECT 'hazard','sectors_exposed',      cdp_disclosing_org_number, row_order, sectors_exposed_english      FROM `{BQ_DATASET_PROCESSED}.hazard_translated`
  UNION ALL SELECT 'hazard','impacts',              cdp_disclosing_org_number, row_order, impacts_english              FROM `{BQ_DATASET_PROCESSED}.hazard_translated`
  UNION ALL SELECT 'funding','project_area',        cdp_disclosing_org_number, row_order, project_area_english         FROM `{BQ_DATASET_PROCESSED}.funding_translated`
  UNION ALL SELECT 'funding','project_title',       cdp_disclosing_org_number, row_order, project_title_english        FROM `{BQ_DATASET_PROCESSED}.funding_translated`
  UNION ALL SELECT 'funding','finance_status',      cdp_disclosing_org_number, row_order, finance_status_english       FROM `{BQ_DATASET_PROCESSED}.funding_translated`
  UNION ALL SELECT 'funding','finance_model',       cdp_disclosing_org_number, row_order, finance_model_english        FROM `{BQ_DATASET_PROCESSED}.funding_translated`
  UNION ALL SELECT 'funding','project_descirption', cdp_disclosing_org_number, row_order, project_descirption_english  FROM `{BQ_DATASET_PROCESSED}.funding_translated`
)
SELECT
  r.pipeline, r.col_name, r.cdp_disclosing_org_number AS org_id, r.row_order,
  r.raw_text, t.english_text,
  (LENGTH(TRIM(r.raw_text)) <= 9
   AND ARRAY_LENGTH(SPLIT(TRIM(r.raw_text), ' ')) = 1
   AND NOT REGEXP_CONTAINS(r.raw_text, r'[^[:ascii:]]')
   AND LENGTH(REGEXP_REPLACE(r.raw_text, r'[^A-Z]', '')) * 2
       > LENGTH(REGEXP_REPLACE(r.raw_text, r'[^A-Za-z]', '')))
    AS triggered_heuristic,
  REGEXP_CONTAINS(
    r.raw_text,
    r'[A-Z][A-Za-z]*(?:\\.[A-Za-z]+){{2,}}\\.?|(?:[A-Za-z]\\.){{2,}}[A-Za-z]?|[A-Z][A-Za-z0-9]*[A-Z0-9][A-Za-z0-9]*(?:[/\\-][A-Za-z0-9]+)*'
  ) AS has_acronym_in_raw,
  REGEXP_EXTRACT_ALL(
    r.raw_text,
    r'[A-Z][A-Za-z]*(?:\\.[A-Za-z]+){{2,}}\\.?|(?:[A-Za-z]\\.){{2,}}[A-Za-z]?|[A-Z][A-Za-z0-9]*[A-Z0-9][A-Za-z0-9]*(?:[/\\-][A-Za-z0-9]+)*'
  ) AS acronyms_in_raw,
    ARRAY(
    SELECT a FROM UNNEST(REGEXP_EXTRACT_ALL(r.raw_text,
      r'[A-Z][A-Za-z]*(?:\\.[A-Za-z]+){{2,}}\\.?|(?:[A-Za-z]\\.){{2,}}[A-Za-z]?|[A-Z][A-Za-z0-9]*[A-Z0-9][A-Za-z0-9]*(?:[/\\-][A-Za-z0-9]+)*')) AS a
    WHERE t.english_text IS NULL OR STRPOS(t.english_text, a) = 0
  ) AS acronyms_dropped,
  IFNULL(REGEXP_CONTAINS(t.english_text, r'</?span'), FALSE) AS has_unstripped_span_tag,
  REGEXP_CONTAINS(r.raw_text, r'[^[:ascii:]]')             AS raw_has_non_ascii,
  ARRAY_LENGTH(SPLIT(TRIM(r.raw_text), ' ')) = 1           AS is_single_token,
  LENGTH(r.raw_text)                                       AS length_chars,
  CASE
    WHEN t.english_text IS NULL                            THEN 'output_null'
    WHEN TRIM(t.english_text) = ''                         THEN 'output_empty'
    WHEN r.raw_text = t.english_text                       THEN 'kept_verbatim'
    ELSE                                                        'translated_changed'
  END AS route
FROM raw_unpivoted r
LEFT JOIN translated_unpivoted t USING (pipeline, col_name, cdp_disclosing_org_number, row_order)
"""
bq_client.query(translation_audit_query).result()
print(f"Created {BQ_DATASET_PROCESSED}.translation_audit")

# Summary stats by pipeline
summary_query = f"""
SELECT
  pipeline,
  COUNT(*) AS total_rows,
  COUNTIF(triggered_heuristic)                       AS heuristic_hits,
  COUNTIF(has_acronym_in_raw)                        AS rows_with_acronyms,
  COUNTIF(triggered_heuristic AND raw_has_non_ascii) AS heuristic_overfire_check,
  COUNTIF(route='output_null')   AS output_null,
  COUNTIF(route='output_empty')  AS output_empty,
  COUNTIF(route='kept_verbatim') AS kept_verbatim,
  COUNTIF(route='translated_changed') AS translated_changed
FROM `{BQ_DATASET_PROCESSED}.translation_audit`
GROUP BY pipeline
ORDER BY pipeline
"""
df = bq_client.query(summary_query).to_dataframe()
print(df.to_string(index=False))


Created project-bb4fd058-24e7-4ccb-b06.CSTAR_2025_processed_v2.translation_audit


/Users/laura/Projects/cdp-google/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


pipeline  total_rows  heuristic_hits  rows_with_acronyms  heuristic_overfire_check  output_null  output_empty  kept_verbatim  translated_changed
  action       56411               0                5321                         0            0             0          52419                3992
 funding       15010              17                2442                         0            0             0          12546                2464
  hazard       20769               0                1260                         0            0             0          17523                3246


#### Goal pipeline audit (lightweight)

The goal pipeline aggregates inline (no `goal_translated` intermediate table), so we can't do per-row side-by-side audit without modifying cell 19. This lightweight version compares each raw `Q5_1_1.col2_Adaptation_goal` against the set of `goal_english` values in `fact_goal` for the same org, and flags `triggered_heuristic` from the raw text.

What we get:
- `triggered_heuristic = TRUE  AND preserved_verbatim = TRUE`  → heuristic correctly bypassed translation. ✓
- `triggered_heuristic = TRUE  AND preserved_verbatim = FALSE` → **bug** — heuristic fired but the row got mangled or filtered.
- `triggered_heuristic = FALSE AND preserved_verbatim = TRUE`  → text was detected as English (or translated to itself).
- `triggered_heuristic = FALSE AND preserved_verbatim = FALSE` → row was translated. We don't see the new text without modifying cell 19.

Trade-off: lose the side-by-side text-of-translation for translated rows, gain a no-cell-19-change audit.

In [79]:
# Goal pipeline audit: org-level set comparison (no cell 19 changes needed).

goal_audit_query = f"""
CREATE OR REPLACE TABLE `{BQ_DATASET_PROCESSED}.goal_audit` AS
WITH raw_goals AS (
  SELECT cdp_disclosing_org_number, row_order, TRIM(col2_Adaptation_goal) AS raw_text
    FROM `{BQ_DATASET_RAW}.Q5_1_1`
   WHERE col2_Adaptation_goal IS NOT NULL AND TRIM(col2_Adaptation_goal) != ''
),
final_goal_set AS (
  SELECT cdp_disclosing_org_number, ARRAY_AGG(DISTINCT goal_english) AS goal_english_set, STRING_AGG(goal_english, ' ¶ ') AS goal_english_concat
    FROM `{BQ_DATASET_PROCESSED}.fact_goal`
   WHERE goal_english IS NOT NULL
   GROUP BY cdp_disclosing_org_number
)
SELECT
  rg.cdp_disclosing_org_number AS org_id,
  rg.row_order,
  rg.raw_text,
  rg.raw_text IN UNNEST(COALESCE(fg.goal_english_set, [])) AS preserved_verbatim,
  (LENGTH(TRIM(rg.raw_text)) <= 9
   AND ARRAY_LENGTH(SPLIT(TRIM(rg.raw_text), ' ')) = 1
   AND NOT REGEXP_CONTAINS(rg.raw_text, r'[^[:ascii:]]')
   AND LENGTH(REGEXP_REPLACE(rg.raw_text, r'[^A-Z]', '')) * 2
       > LENGTH(REGEXP_REPLACE(rg.raw_text, r'[^A-Za-z]', '')))                AS triggered_heuristic,
  REGEXP_CONTAINS(rg.raw_text,
    r'[A-Z][A-Za-z]*(?:\\.[A-Za-z]+){{2,}}\\.?|(?:[A-Za-z]\\.){{2,}}[A-Za-z]?|[A-Z][A-Za-z0-9]*[A-Z0-9][A-Za-z0-9]*(?:[/\\-][A-Za-z0-9]+)*'
  ) AS has_acronym_in_raw,
  REGEXP_EXTRACT_ALL(rg.raw_text,
    r'[A-Z][A-Za-z]*(?:\\.[A-Za-z]+){{2,}}\\.?|(?:[A-Za-z]\\.){{2,}}[A-Za-z]?|[A-Z][A-Za-z0-9]*[A-Z0-9][A-Za-z0-9]*(?:[/\\-][A-Za-z0-9]+)*'
  ) AS acronyms_in_raw,
    ARRAY(
    SELECT a FROM UNNEST(REGEXP_EXTRACT_ALL(rg.raw_text,
      r'[A-Z][A-Za-z]*(?:\\.[A-Za-z]+){{2,}}\\.?|(?:[A-Za-z]\\.){{2,}}[A-Za-z]?|[A-Z][A-Za-z0-9]*[A-Z0-9][A-Za-z0-9]*(?:[/\\-][A-Za-z0-9]+)*')) AS a
    WHERE fg.goal_english_concat IS NULL OR STRPOS(fg.goal_english_concat, a) = 0
  ) AS acronyms_dropped,
  IFNULL(REGEXP_CONTAINS(fg.goal_english_concat, r'</?span'), FALSE) AS has_unstripped_span_tag,
  REGEXP_CONTAINS(rg.raw_text, r'[^[:ascii:]]')  AS raw_has_non_ascii,
  LENGTH(rg.raw_text)                            AS length_chars
FROM raw_goals rg
LEFT JOIN final_goal_set fg ON rg.cdp_disclosing_org_number = fg.cdp_disclosing_org_number
"""
bq_client.query(goal_audit_query).result()
print(f"Created {BQ_DATASET_PROCESSED}.goal_audit")

# Summary breakdown
goal_summary_query = f"""
SELECT
  triggered_heuristic, preserved_verbatim,
  COUNT(*) AS row_count
FROM `{BQ_DATASET_PROCESSED}.goal_audit`
GROUP BY 1, 2
ORDER BY 1 DESC, 2 DESC
"""
print()
print("=== Goal audit: routing summary ===")
print(bq_client.query(goal_summary_query).to_dataframe().to_string(index=False))

# Safety check: heuristic-triggered goals that did NOT survive verbatim
print()
print("=== ⚠ Goals where heuristic fired but raw text NOT preserved (should be 0 rows) ===")
print(bq_client.query(f"""
SELECT org_id, row_order, raw_text, length_chars
FROM `{BQ_DATASET_PROCESSED}.goal_audit`
WHERE triggered_heuristic AND NOT preserved_verbatim
ORDER BY org_id, row_order
""").to_dataframe().to_string(index=False))


Created project-bb4fd058-24e7-4ccb-b06.CSTAR_2025_processed_v2.goal_audit

=== Goal audit: routing summary ===


/Users/laura/Projects/cdp-google/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


 triggered_heuristic  preserved_verbatim  row_count
               False                True       3035
               False               False       1552

=== ⚠ Goals where heuristic fired but raw text NOT preserved (should be 0 rows) ===
Empty DataFrame
Columns: [org_id, row_order, raw_text, length_chars]
Index: []


/Users/laura/Projects/cdp-google/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
